# Masked ST-Transformer BIG V3 / V4 — PEMS-BAY Benchmark (Anti-Cheat v4)
### Dataset: PEMS-BAY (325 nodes, 80% sparsity, 5 seeds). Zero-as-missing convention.

**Anti-cheat audit v4** — every model receives *only the same 20% observed sensor readings*
as input. No model sees the held-out 80% or future time steps.

### All fixes applied

| # | Fix | Models affected |
|---|-----|-----------------|
| v2-1 | BiLSTM bidirectional→forward | BiLSTM |
| v2-2 | BRITS backward GRU removed | BRITS-lite |
| v2-3 | SAITS/ASTGCN causal attention mask | SAITS-lite, ASTGCN-lite |
| v3-1 | Unified numpy RNG for eval masks | ALL models |
| v3-2 | Eval chunk size = training window (48) | Our model |
| v4-1 | Node embedding added | MLP, SAITS-lite |
| v4-2 | KNN masked-distance (only observed dims) | KNN |
| v4-3 | Graph models: HA fill for unobserved nodes | DCRNN, GWN, ASTGCN |

| v4-5 | Staleness feature (steps-since-last-obs / 48) | V4 |
| v4-6 | Soft LOCF: EMA(0.95) toward HA prior | V4 |
| v4-7 | Masking curriculum 60% to 80% over 600 epochs | V4 |
| v4-8 | Larger model: hidden 128, 6 layers, 1500 epochs | V4 |

**v4-1 rationale:** at 80% sparsity, unobserved nodes receive input `[0, 0, sin, cos]`.
Without a node embedding, MLP and SAITS cannot distinguish sensors and collapse to
a global prediction — explaining their worse-than-HA results.

**v4-3 rationale:** graph diffusion was spreading masked zeros into neighbouring nodes,
corrupting the spatial signal. Filling with the HA prior gives a neutral, informative
starting point while the mask feature still tells the model which nodes are real.


In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory ready.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import glob
import pickle
import urllib.request
import warnings

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_NAME = 'PEMS-BAY'
SPARSITY = 0.80
BATCH_TIME = 48
HIDDEN_DIM = 96
N_LAYERS = 5
N_HEADS = 4
DROPOUT = 0.1
TRAIN_EPOCHS = 1200
STEPS_PER_DAY = 288
EVAL_SEEDS = [42, 43, 44, 45, 46]
HUBER_BETA = 1.0

PEMSBAY_CSV_URL = "https://zenodo.org/records/5146275/files/PEMS-BAY.csv?download=1"
PEMSBAY_ADJ_URL = "https://zenodo.org/records/5146275/files/adj_mx_bay.pkl?download=1"

def find_file(candidates, search_roots=('.', '/kaggle/input', '/kaggle/working')):
    cand_lower = [c.lower() for c in candidates]
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, '**', '*'), recursive=True):
            if os.path.isfile(path) and os.path.basename(path).lower() in cand_lower:
                return path
    return None

def download_if_missing(url, dest):
    if not os.path.exists(dest):
        print(f"Downloading {dest} from {url} ...")
        urllib.request.urlretrieve(url, dest)
    return dest

def load_speed_array():
    """Return speed_raw [T, N] np.float32 from .h5 or .csv."""
    h5 = find_file(['pems-bay.h5', 'PEMS-BAY.h5'])
    if h5 is not None:
        print(f"  H5: {h5}")
        df = pd.read_hdf(h5)
        return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
    csv = find_file(['pems-bay.csv', 'PEMS-BAY.csv'])
    if csv is None:
        csv = download_if_missing(PEMSBAY_CSV_URL, 'PEMS-BAY.csv')
    print(f"  CSV (timeseries): {csv}")
    df = pd.read_csv(csv, index_col=0)
    return np.nan_to_num(df.values.astype(np.float32), nan=0.0)

def load_adjacency(num_nodes):
    """Return adjacency matrix [N, N] from DCRNN-format .pkl. PEMS-BAY-specific filenames."""
    pkl = find_file(['adj_mx_bay.pkl', 'adj_mx_pems_bay.pkl'])
    if pkl is None:
        pkl = download_if_missing(PEMSBAY_ADJ_URL, 'adj_mx_bay.pkl')
    print(f"  Adj PKL: {pkl}")
    with open(pkl, 'rb') as f:
        obj = pickle.load(f, encoding='latin1')
    adj_mx = obj[2] if isinstance(obj, (list, tuple)) and len(obj) >= 3 else obj
    adj_mx = np.asarray(adj_mx, dtype=np.float32)
    if adj_mx.shape[0] != num_nodes:
        raise ValueError(
            f"Adjacency shape {adj_mx.shape} doesn't match data ({num_nodes} nodes). "
            f"Wrong pickle picked up -- delete cached adj_mx*.pkl and re-run."
        )
    adj = (adj_mx > 0.1).astype(np.float32)
    np.fill_diagonal(adj, 0)
    return adj

print(f"Device: {device} | Dataset: {DATASET_NAME} | Target Sparsity: {SPARSITY*100}%")

In [ ]:
class STBlock(nn.Module):
    def __init__(self, hidden, n_heads, ff_mult=2, dropout=0.0):
        super().__init__()
        self.temp = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.spat = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)

    def forward(self, h, spatial_pad_mask=None):
        B, N, T, H = h.shape
        cm = torch.triu(torch.full((T, T), float('-inf'), device=h.device), diagonal=1)
        h = self.temp(h.reshape(B * N, T, H), src_mask=cm).reshape(B, N, T, H)
        h = h.permute(0, 2, 1, 3).contiguous().reshape(B * T, N, H)
        if spatial_pad_mask is not None:
            h = self.spat(h, src_key_padding_mask=spatial_pad_mask)
        else:
            h = self.spat(h)
        h = h.reshape(B, T, N, H).permute(0, 2, 1, 3).contiguous()
        return h


class MaskedSTTransformerV3(nn.Module):
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=64, n_heads=4, n_layers=3, max_T=288, dropout=0.0,
                 nmean_trust_thresh=0.05):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.nmean_trust_thresh = nmean_trust_thresh
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)  # [N]
        self.register_buffer('node_stds', node_stds_t)    # [N]

        # 6 input features: x, m, t_sin, t_cos, n_mean, locf
        self.in_proj = nn.Linear(6, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb = nn.Parameter(torch.randn(max_T, hidden) * 0.02)

        self.blocks = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)

        self.meta_gate = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Linear(hidden, 1)
        self.last_alpha = None
        self.last_nmean_trust_frac = None  # diagnostic: fraction of positions where n_mean is trusted

    def _compute_locf(self, x, m):
        B, N, T = x.shape
        locf = torch.zeros_like(x)
        current_val = torch.zeros(B, N, device=x.device)  # default to mean (z-score = 0)
        for t in range(T):
            obs_t = x[:, :, t]
            mask_t = m[:, :, t]
            current_val = torch.where(mask_t > 0.5, obs_t, current_val)
            locf[:, :, t] = current_val
        return locf

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v = self.node_stds.view(1, -1, 1)

        # 1. Compute causal LOCF on the fly
        locf = self._compute_locf(x, m)
        locf_kmh = locf * std_v + mean_v

        # 2. Compute spatial neighbor mean over robust LOCF speeds
        adj_x_kmh = torch.matmul(self.adj_static, locf_kmh)
        adj_m = self.adj_static.sum(dim=-1, keepdim=True).view(1, -1, 1)
        n_mean_kmh = adj_x_kmh / (adj_m + 1e-6)
        n_mean = (n_mean_kmh - mean_v) / std_v
        
        self.last_nmean_trust_frac = torch.tensor(1.0, device=x.device) # always trusted

        # Stack 6 input features: x, m, t_sin, t_cos, n_mean, locf
        feat = torch.stack([x, m, t_sin, t_cos, n_mean, locf], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)

        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        residual = self.residual_head(h).squeeze(-1)
        
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()
        
        # 3. Add learned residual correction directly to LOCF-imputed speed!
        return locf + alpha * residual

class MaskedSTTransformerV4(nn.Module):
    """V4: staleness + soft_locf features, hidden=128, n_layers=6, masking curriculum.
    Input features (8): x, m, t_sin, t_cos, n_mean, locf, staleness, soft_locf
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)

        self.in_proj  = nn.Linear(8, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks     = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                         for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Linear(hidden, 1)
        self.last_alpha    = None

    def _compute_causal_signals(self, x, m, ha_prior):
        """Causal LOCF, staleness (steps-since-obs/48), soft_locf (EMA toward HA)."""
        B, N, T = x.shape
        locf      = torch.zeros_like(x)
        staleness = torch.zeros_like(x)
        soft_locf = torch.zeros_like(x)
        cur_locf  = torch.zeros(B, N, device=x.device)
        cur_stale = torch.zeros(B, N, device=x.device)
        cur_soft  = torch.zeros(B, N, device=x.device)
        decay = self.soft_locf_decay
        for t in range(T):
            obs_t  = x[:, :, t]
            mask_t = m[:, :, t]
            ha_t   = ha_prior[:, :, t]
            obs    = mask_t > 0.5
            cur_locf  = torch.where(obs, obs_t, cur_locf)
            cur_stale = torch.where(obs, torch.zeros_like(cur_stale), cur_stale + 1.0)
            cur_soft  = torch.where(obs, obs_t,
                                    decay * cur_soft + (1.0 - decay) * ha_t)
            locf     [:, :, t] = cur_locf
            staleness[:, :, t] = cur_stale / 48.0
            soft_locf[:, :, t] = cur_soft
        return locf, staleness, soft_locf

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)
        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh  = locf * std_v + mean_v
        adj_x_kmh = torch.matmul(self.adj_static, locf_kmh)
        adj_deg   = self.adj_static.sum(dim=-1, keepdim=True).view(1, -1, 1)
        n_mean    = (adj_x_kmh / (adj_deg + 1e-6) - mean_v) / std_v
        feat = torch.stack([x, m, t_sin, t_cos, n_mean, locf, staleness, soft_locf], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)
        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)
        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()
        return locf + alpha * residual


class MaskedSTTransformerV5(nn.Module):
    """V5 improvements over V4:
    1. Output base: soft_locf + alpha*residual (was locf). Correction learns
       from a better anchor; smaller residual target for stale positions.
    2. 2-hop neighbor mean (n_mean_2hop) as 9th input feature. At 80%
       sparsity each node has ~1.5 observed 1-hop neighbours; 2-hop gives ~5.
    3. Deeper residual head: hidden -> hidden//2 -> 1 (was linear).
    4. Training: 2000 epochs, batch size 4.
    Input features (9): x, m, t_sin, t_cos, n_mean, locf, staleness, soft_locf, n_mean_2hop
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)

        # Precompute row-normalised 2-hop adjacency (no self-loops)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop',
                                 adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        self.in_proj  = nn.Linear(9, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1))
        self.last_alpha = None

    def _compute_causal_signals(self, x, m, ha_prior):
        """Causal LOCF, staleness (steps-since-obs/48), soft_locf (EMA toward HA)."""
        B, N, T = x.shape
        locf      = torch.zeros_like(x)
        staleness = torch.zeros_like(x)
        soft_locf = torch.zeros_like(x)
        cur_locf  = torch.zeros(B, N, device=x.device)
        cur_stale = torch.zeros(B, N, device=x.device)
        cur_soft  = torch.zeros(B, N, device=x.device)
        decay = self.soft_locf_decay
        for t in range(T):
            obs_t  = x[:, :, t]
            mask_t = m[:, :, t]
            ha_t   = ha_prior[:, :, t]
            obs    = mask_t > 0.5
            cur_locf  = torch.where(obs, obs_t, cur_locf)
            cur_stale = torch.where(obs, torch.zeros_like(cur_stale), cur_stale + 1.0)
            cur_soft  = torch.where(obs, obs_t,
                                    decay * cur_soft + (1.0 - decay) * ha_t)
            locf     [:, :, t] = cur_locf
            staleness[:, :, t] = cur_stale / 48.0
            soft_locf[:, :, t] = cur_soft
        return locf, staleness, soft_locf

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)

        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v

        # 1-hop neighbour mean (z-scored)
        adj_deg  = self.adj_static.sum(dim=-1, keepdim=True).view(1, -1, 1)
        n_mean   = (torch.matmul(self.adj_static, locf_kmh) / (adj_deg + 1e-6) - mean_v) / std_v

        # 2-hop neighbour mean (z-scored) -- richer spatial context at high sparsity
        n_mean_2hop = (torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v

        feat = torch.stack([x, m, t_sin, t_cos, n_mean, locf,
                            staleness, soft_locf, n_mean_2hop], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()

        # Base: soft_locf (not locf) -- correction needs to close a smaller gap
        return soft_locf + alpha * residual


In [ ]:
print("Loading data...")
speed_raw = load_speed_array()[:5000]
NUM_NODES = speed_raw.shape[1]
TRAIN_END = 4000
print(f"  Detected: T={speed_raw.shape[0]}, N={NUM_NODES}")

# PEMS-BAY: 0 = missing reading. Build a validity mask and apply it everywhere.
valid_raw = (speed_raw > 0).astype(np.float32)
print(f"  Valid fraction: {valid_raw.mean():.3f}")

# Per-node mean/std over valid entries only
node_means = np.zeros(NUM_NODES, dtype=np.float32)
node_stds = np.ones(NUM_NODES, dtype=np.float32)
for n in range(NUM_NODES):
    vals = speed_raw[:TRAIN_END, n][valid_raw[:TRAIN_END, n] > 0]
    if len(vals) > 0:
        node_means[n] = vals.mean()
        node_stds[n] = vals.std() + 1e-8

speed_norm = (speed_raw - node_means) / node_stds

# HA prior per (node, time-of-day), averaged over valid entries only
slot_idx = np.arange(len(speed_norm)) % STEPS_PER_DAY
tod_mean = np.zeros((NUM_NODES, STEPS_PER_DAY), dtype=np.float32)
for s in range(STEPS_PER_DAY):
    sel = slot_idx[:TRAIN_END] == s
    sub_data = speed_norm[:TRAIN_END][sel]
    sub_valid = valid_raw[:TRAIN_END][sel]
    sums = (sub_data * sub_valid).sum(axis=0)
    cnts = sub_valid.sum(axis=0) + 1e-8
    tod_mean[:, s] = sums / cnts

ha_prior = torch.tensor(tod_mean[:, slot_idx].T, dtype=torch.float32).to(device)
speed_gpu = torch.tensor(speed_norm, dtype=torch.float32).to(device)
valid_gpu = torch.tensor(valid_raw, dtype=torch.float32).to(device)
node_means_t = torch.tensor(node_means, dtype=torch.float32).to(device)
node_stds_t = torch.tensor(node_stds, dtype=torch.float32).to(device)

adj = load_adjacency(NUM_NODES)
D = np.diag(1.0 / np.sqrt(adj.sum(axis=1) + 1e-8))
adj_norm = D @ adj @ D
A_t = torch.tensor(adj_norm, dtype=torch.float32).to(device)

print(f"Data and adjacency ready. NUM_NODES={NUM_NODES}, avg degree={adj.sum(1).mean():.2f}, edges={int(adj.sum())}")

## Baseline Models for Comparison (Fair, Causal Evaluation)

All models receive **only the 20% observed sensor readings** (`x * m_eff`) as input.
No model has access to the held-out 80% or to future time steps.
Models marked **[fixed]** had future-leakage bugs that have now been corrected.

| # | Model | Family | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 1 | Historical Average (HA) | Statistical | No | ✓ |
| 2 | LOCF | Statistical | No | ✓ |
| 3 | Global Mean | Statistical | No | ✓ |
| 4 | Ridge Regression | Linear | No | ✓ |
| 5 | KNN Imputer (spatial, train-set neighbors only) | Non-param | No | ✓ |
| 6 | MLP | Neural | No | ✓ |
| 7 | LSTM | RNN | No | ✓ |
| 8 | BiLSTM → 2L-LSTM **[fixed]** | RNN | No | ✓ |
| 9 | GRU | RNN | No | ✓ |
| 10 | TCN (causal dilated) | Conv | No | ✓ |
| 11 | SAITS-lite **[fixed]** | Transformer | No | ✓ |
| 12 | BRITS-lite **[fixed]** | RNN+decay | No | ✓ |
| 13 | DCRNN-lite | GNN+RNN | Yes | ✓ |
| 14 | ASTGCN-lite **[fixed]** | GNN+Attn | Yes | ✓ |
| 15 | GWN-lite | GNN+TCN | Yes | ✓ |
| 16 | **MaskedSTTransformerV3-FIXED (ours)** | ST-Attn | Yes | ✓ |


In [ ]:
import numpy as np, torch

RESULTS = {}

# ── Shared mask generator (ALL models must use this) ──────────────────────
# Uses numpy default_rng so masks are identical regardless of torch state.
def make_eval_mask_np(seed, EL, N):
    """Return bool mask [EL, N] — True = observed (20%), False = held-out (80%)."""
    rng = np.random.default_rng(seed)
    return (rng.random((EL, N)) > SPARSITY).astype(np.float32)  # [T, N]

# ── Shared eval helper (numpy, for non-graph models) ──────────────────────
def eval_fn(pred_fn, label):
    ES, EL = 4500, 450
    x_ev = speed_norm[ES:ES+EL]   # [T,N]
    v_ev = valid_raw [ES:ES+EL]
    maes = []
    for seed in EVAL_SEEDS:
        m_ev = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
        sm   = (m_ev == 0) & (v_ev > 0)
        if not sm.any(): continue
        idx  = np.arange(ES, ES+EL)
        p    = np.clip(pred_fn(x_ev.T, v_ev.T, m_ev.T, idx), 0, 120)
        t    = np.clip(x_ev.T * node_stds[:,None] + node_means[:,None], 0, 120)
        maes.append(np.abs(p[sm.T] - t[sm.T]).mean())
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr

# 1. Historical Average
eval_fn(lambda xn,vn,mn,idx: tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None],
        "1. Historical Average (HA)")

# 2. LOCF
def locf(xn, vn, mn, idx):
    N, T = xn.shape
    out  = tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None]
    last = out[:,0].copy()
    for t in range(T):
        obs  = (mn[:,t]>0)&(vn[:,t]>0)
        last = np.where(obs, xn[:,t]*node_stds+node_means, last)
        out[:,t] = last
    return out
eval_fn(locf, "2. LOCF (Last-Obs Carried Forward)")

# 3. Global mean
eval_fn(lambda xn,vn,mn,idx: np.broadcast_to(node_means[:,None],xn.shape).copy(),
        "3. Global Mean (per-node train mean)")


In [ ]:
from sklearn.linear_model import Ridge
import warnings; warnings.filterwarnings("ignore")

# 4. Ridge Regression (unchanged)
print("Fitting Ridge regressors...")
ridge_models = []
for n in range(NUM_NODES):
    si  = slot_idx[:TRAIN_END]
    Xtr = np.column_stack([np.sin(2*np.pi*si/STEPS_PER_DAY),
                           np.cos(2*np.pi*si/STEPS_PER_DAY),
                           tod_mean[n, si]])
    ytr = speed_norm[:TRAIN_END, n]
    mk  = valid_raw[:TRAIN_END, n] > 0
    clf = Ridge(alpha=1.0); clf.fit(Xtr[mk], ytr[mk])
    ridge_models.append(clf)

def ridge(xn, vn, mn, idx):
    N, T = xn.shape; out = np.zeros((N,T),dtype=np.float32)
    for n in range(N):
        Xn = np.column_stack([np.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              np.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              tod_mean[n, idx%STEPS_PER_DAY]])
        out[n] = ridge_models[n].predict(Xn)*node_stds[n]+node_means[n]
    return out
eval_fn(ridge, "4. Node-wise Ridge Regression")

# 5. KNN Imputer — causal spatial imputation with masked Euclidean distance.
#
# Why the previous fix failed: at 80% sparsity, a 325-dim query has ~260 zeros
# (unobserved sensors set to 0). Standard Euclidean distance treats those zeros
# as signal, so neighbours are found based on the zero pattern, not speed values.
#
# Fix: compute distance only over the ~65 observed dimensions, normalised by
# the number of shared observed sensors. This is the standard 'missing-value
# aware' nearest-neighbour approach used in the imputation literature.
# Causal: each timestep is treated independently (spatial, not temporal KNN).
print("Fitting KNN (masked-distance spatial imputer)...")
tr_norm = speed_norm[:TRAIN_END].copy()           # [T_tr, N] z-scored
tr_valid = (valid_raw[:TRAIN_END] > 0).astype(np.float32)  # [T_tr, N]
# Keep only training rows with >=10% valid sensors
enough    = tr_valid.mean(axis=1) >= 0.10
tr_ref    = tr_norm[enough]          # [M, N]
tv_ref    = tr_valid[enough]         # [M, N]  1=valid, 0=missing
K_NEIGH   = 5

def knn_masked(xn, vn, mn, idx):
    """Causal spatial KNN with masked Euclidean distance.
    xn,vn,mn: [N,T] numpy arrays (z-scored / valid / observed-mask).
    Returns [N,T] in km/h.
    Each timestep is solved independently — no future used.
    """
    N, T = xn.shape
    out = (tod_mean[:, idx % STEPS_PER_DAY] * node_stds[:, None]
           + node_means[:, None]).copy()   # HA fallback [N,T]
    for t in range(T):
        obs = (mn[:, t] > 0) & (vn[:, t] > 0)   # [N] bool — truly observed
        if obs.sum() < 2:
            continue   # too few observed sensors — keep HA
        q_vals = xn[:, t]          # [N] z-scored query row

        # Masked squared distance: average over shared valid dimensions only
        # shared[m,n] = 1 if both query and ref row m have sensor n valid
        shared = tv_ref * obs.astype(np.float32)  # [M, N]
        n_shared = shared.sum(axis=1) + 1e-8      # [M]
        diff = (tr_ref - q_vals) * shared         # [M, N] — zero out unshared
        dist = (diff ** 2).sum(axis=1) / n_shared  # [M] mean-sq dist over shared

        # k nearest neighbours
        knn_idx = np.argpartition(dist, K_NEIGH)[:K_NEIGH]  # [K]
        neigh_v  = tr_ref[knn_idx]    # [K, N] z-scored
        neigh_ok = tv_ref[knn_idx]    # [K, N] validity

        # Fill: observed sensors keep their value; missing → weighted avg of neighbours
        neigh_sum = (neigh_v * neigh_ok).sum(0)     # [N]
        neigh_cnt = neigh_ok.sum(0) + 1e-8          # [N]
        imputed   = neigh_sum / neigh_cnt            # [N] z-scored
        row_norm  = np.where(obs, q_vals, imputed)   # [N] z-scored
        out[:, t] = np.clip(row_norm * node_stds + node_means, 0, 120)
    return out

eval_fn(knn_masked, "5. KNN Imputer (k=5, masked-dist)")


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

NE, NH, NLR, NBT = 1200, 64, 1e-3, 48

def train_nw(cls, label, **kw):
    net = cls(4, NH, **kw).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=NLR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NE)
    xg, vg = speed_gpu[:TRAIN_END], valid_gpu[:TRAIN_END]
    T_tr, N = xg.shape
    for ep in range(1, NE+1):
        net.train()
        t0 = np.random.randint(0, T_tr-NBT)
        xb, vb = xg[t0:t0+NBT], vg[t0:t0+NBT]
        idx = torch.arange(t0, t0+NBT, device=device)
        s   = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        c   = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        mb  = (torch.rand(NBT, N, device=device) > SPARSITY).float()
        me  = mb * vb; xi = xb * me
        feat = torch.stack([xi.T, me.T,
            s.unsqueeze(0).expand(N, -1),
            c.unsqueeze(0).expand(N, -1)], dim=-1)   # [N,T,4]
        pred = net(feat).squeeze(-1)                  # [N,T]
        lm = (mb.T == 0) & (vb.T > 0)
        if not lm.any(): continue
        loss = F.smooth_l1_loss(pred[lm], xb.T[lm])  # z-scored targets
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_nw(net, label):
    """Uses make_eval_mask_np — identical masks across all models."""
    net.eval()
    ES, EL = 4500, 450
    xev = speed_gpu[ES:ES+EL]; vev = valid_gpu[ES:ES+EL]
    idx = torch.arange(ES, ES+EL, device=device)
    s = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    c = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    st  = torch.tensor(node_stds,  device=device)
    mn_ = torch.tensor(node_means, device=device)
    maes = []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(
                make_eval_mask_np(seed, EL, NUM_NODES), device=device)  # [T,N]
            me   = m_ev * vev
            feat = torch.stack([
                (xev * me).T, me.T,
                s.unsqueeze(0).expand(NUM_NODES, -1),
                c.unsqueeze(0).expand(NUM_NODES, -1)], dim=-1)  # [N,T,4]
            pred = net(feat).squeeze(-1)               # [N,T]
            pk   = (pred * st[:, None] + mn_[:, None]).clamp(0, 120)
            tk   = (xev.T * st[:, None] + mn_[:, None]).clamp(0, 120)
            sm   = (m_ev.T == 0) & (vev.T > 0)
            maes.append(torch.abs(pk[sm] - tk[sm]).mean().item())
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr

# ── MLP with node embedding ───────────────────────────────────────────────
# ROOT CAUSE FIX: at 80% sparsity, an unobserved node's input is [0,0,sin,cos]
# — indistinguishable from any other unobserved node at the same timestep.
# Without a node identity, the MLP predicts the same value for all missing
# nodes, behaving like a global mean. Adding a learned node embedding gives
# each sensor a unique fingerprint even when its value is masked out.
class MLP(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        self.net = nn.Sequential(
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H * 2), nn.GELU(),
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H),     nn.GELU(),
            nn.Linear(H, 1))
        self._n = n_nodes

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        nids = torch.arange(N, device=x.device)
        ne   = self.node_emb(nids).unsqueeze(1).expand(N, T, -1)  # [N,T,H]
        hf   = self.proj(x)                                        # [N,T,H]
        return self.net(torch.cat([hf, ne], dim=-1))               # [N,T,1]

class LSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.LSTM(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

# FIXED (v2): bidirectional=True saw future; replaced with 2-layer forward LSTM.
class BiLSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.r=nn.LSTM(F, H, num_layers=2, batch_first=True, dropout=0.1)
        self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

class GRUNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.GRU(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

eval_nw(train_nw(MLP,    "MLP",    n_nodes=NUM_NODES), "6.  MLP (per-node + node-emb)")
eval_nw(train_nw(LSTM,   "LSTM"),                      "7.  LSTM (per-node)")
eval_nw(train_nw(BiLSTM, "BiLSTM"),                    "8.  BiLSTM->2L-LSTM (causal, fixed)")
eval_nw(train_nw(GRUNet, "GRU"),                       "9.  GRU (per-node)")


In [ ]:
# 10. TCN — causal dilated convolutions: no fix needed.
class CausalConv(nn.Module):
    def __init__(self,c,k,d):
        super().__init__(); self.p=(k-1)*d; self.c=nn.Conv1d(c,c,k,dilation=d)
    def forward(self,x): return self.c(F.pad(x,(self.p,0)))

class TCNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([CausalConv(H,3,2**i) for i in range(4)])
        self.norms=nn.ModuleList([nn.LayerNorm(H) for _ in range(4)])
        self.head=nn.Linear(H,1)
    def forward(self,x):
        h=self.proj(x).permute(0,2,1)
        for cv,nm in zip(self.convs,self.norms):
            r=h; h=F.gelu(nm(cv(h).permute(0,2,1))).permute(0,2,1)+r
        return self.head(h.permute(0,2,1))
eval_nw(train_nw(TCNet,"TCN"), "10. TCN (causal dilated, per-node)")

# 11. SAITS-lite — FIXED v2 (causal mask) + FIXED v4 (node embedding).
# ROOT CAUSE FIX: same as MLP — at 80% sparsity, unobserved nodes all have
# input [0,0,sin,cos]. Without node identity the transformer cannot distinguish
# sensors and collapses to a global prediction. Node embedding added.
class SAITSLite(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        kw = dict(nhead=4, dim_feedforward=H*2, dropout=0.1,
                  activation='gelu', batch_first=True, norm_first=True)
        self.a1 = nn.TransformerEncoderLayer(H, **kw)
        self.a2 = nn.TransformerEncoderLayer(H, **kw)
        self.h1 = nn.Linear(H, 1); self.h2 = nn.Linear(H, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self._n = n_nodes

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        cm  = self._cmask(T, x.device)
        ne  = self.node_emb(torch.arange(N, device=x.device))  # [N, H]
        h   = self.proj(x) + ne.unsqueeze(1)                   # [N, T, H]
        h1  = self.a1(h,  src_mask=cm)
        h2  = self.a2(h1, src_mask=cm)
        a   = torch.sigmoid(self.alpha)
        return a * self.h1(h1) + (1-a) * self.h2(h2)

eval_nw(train_nw(SAITSLite, "SAITS", n_nodes=NUM_NODES),
        "11. SAITS-lite (causal Attn + node-emb, fixed)")

# 12. BRITS-lite — forward GRU only (fixed v2). No node-emb needed:
# GRU hidden state h carries node-specific temporal history, so the model
# accumulates a unique fingerprint per node through recurrence.
class BRITSLite(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.H=H
        self.gf   = nn.GRU(F*2, H, batch_first=True)
        self.impf  = nn.Linear(H, F)
        self.head  = nn.Linear(H, 1)

    def _run(self,x,m,gru,imp):
        N,T,Ff=x.shape
        h=torch.zeros(1,N,self.H,device=x.device); outs=[]
        for t in range(T):
            xh  = imp(h.squeeze(0))
            xc  = m[:,t,:]*x[:,t,:] + (1-m[:,t,:])*xh
            o,h = gru(torch.cat([xc, m[:,t,:]], -1).unsqueeze(1), h)
            outs.append(o)
        return torch.cat(outs, 1)

    def forward(self,x):
        m  = x[:,:,1:2].expand_as(x)
        hf = self._run(x, m, self.gf, self.impf)
        return self.head(hf)
eval_nw(train_nw(BRITSLite,"BRITS"), "12. BRITS-lite (forward GRU only, fixed)")


In [ ]:
# Graph baselines: feat [B,N,T,F] + adj A -> [B,N,T]
GE, GH, GLR = 1200, 64, 1e-3

def train_g(net, label):
    # ROOT CAUSE FIX for graph models: unobserved nodes previously had x=0
    # fed into graph diffusion, spreading zeros into neighbouring nodes'
    # representations. Fix: fill unobserved node values with HA prior
    # (z-scored). The mask feature still tells the model which nodes are real.
    opt=torch.optim.Adam(net.parameters(),lr=GLR)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=GE)
    xg,vg=speed_gpu[:TRAIN_END],valid_gpu[:TRAIN_END]; T_tr,N=xg.shape
    for ep in range(1,GE+1):
        net.train()
        t0=np.random.randint(0,T_tr-48)
        xb=xg[t0:t0+48].T.unsqueeze(0); vb=vg[t0:t0+48].T.unsqueeze(0)
        ha=ha_prior[t0:t0+48].T.unsqueeze(0)   # [1,N,48] z-scored HA prior
        idx=torch.arange(t0,t0+48,device=device)
        s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        mb=(torch.rand(1,N,48,device=device)>SPARSITY).float(); me=mb*vb
        # ▶ FIX: unobserved positions filled with HA (not 0) before graph diffusion
        xi = xb*me + ha*(1-me)   # observed: real value; unobserved: HA prior
        feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
        pred=net(feat,A_t)
        lm=(mb==0)&(vb>0)
        if not lm.any(): continue
        loss=F.smooth_l1_loss(pred[lm],xb[lm])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_g(net, label):
    net.eval()
    ES,EL=4500,450
    xev=speed_gpu[ES:ES+EL].T.unsqueeze(0); vev=valid_gpu[ES:ES+EL].T.unsqueeze(0)
    hae=ha_prior[ES:ES+EL].T.unsqueeze(0)   # [1,N,EL] HA prior for fill
    idx=torch.arange(ES,ES+EL,device=device)
    s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st=torch.tensor(node_stds,device=device).view(1,-1,1)
    mn_=torch.tensor(node_means,device=device).view(1,-1,1)
    maes=[]
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_np = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
            mev=torch.tensor(m_np, device=device).T.unsqueeze(0)  # [1,N,T]
            me=mev*vev
            # ▶ FIX: fill unobserved with HA prior (not 0) before graph diffusion
            xi = xev*me + hae*(1-me)
            feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
            pred=net(feat,A_t)
            pk=(pred*st+mn_).clamp(0,120); tk=(xev*st+mn_).clamp(0,120)
            sm=(mev==0)&(vev>0)
            maes.append(torch.abs(pk[sm]-tk[sm]).mean().item())
    arr=np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label]=arr

class DiffGCN(nn.Module):
    def __init__(self,i,o,K=2):
        super().__init__(); self.K=K; self.l=nn.Linear((K+1)*i,o)
    def forward(self,x,A):
        out=[x]; Ax=x
        for _ in range(self.K): Ax=torch.matmul(A.unsqueeze(0),Ax); out.append(Ax)
        return self.l(torch.cat(out,-1))

# 13. DCRNN-lite — forward GRU + graph diffusion: strictly causal, no fix needed.
class DCRNNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__(); self.H=H
        self.gr=DiffGCN(F+H,H); self.gu=DiffGCN(F+H,H); self.gc=DiffGCN(F+H,H)
        self.head=nn.Linear(H,1)
    def _step(self,x,h,A):
        xu=torch.cat([x,h],-1)
        r=torch.sigmoid(self.gr(xu,A)); u=torch.sigmoid(self.gu(xu,A))
        c=torch.tanh(self.gc(torch.cat([x,r*h],-1),A))
        return u*h+(1-u)*c
    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=torch.zeros(B,N,self.H,device=feat.device); outs=[]
        for t in range(T): h=self._step(feat[:,:,t,:],h,A); outs.append(self.head(h))
        return torch.stack(outs,2).squeeze(-1)
dcrnn=DCRNNLite(H=GH).to(device); train_g(dcrnn,"DCRNN-lite"); eval_g(dcrnn,"13. DCRNN-lite (DiffGCN + GRU)")

# 14. ASTGCN-lite — FIXED: added causal mask to temporal self-attention.
# Original used full attention with no mask → position t attended to t+k (future leakage).
# Fix: same upper-triangular -inf causal mask as SAITS fix above.
class ASTGCNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__()
        self.proj=nn.Linear(F,H); self.gcn=DiffGCN(H,H)
        self.ta=nn.TransformerEncoderLayer(H,4,H*2,dropout=0.1,
            activation="gelu",batch_first=True,norm_first=True)
        self.head=nn.Linear(H,1)

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=self.proj(feat)
        hs=h.permute(0,2,1,3).reshape(B*T,N,-1)
        hs=self.gcn(hs,A).reshape(B,T,N,-1).permute(0,2,1,3)
        h=h+hs; cm=self._cmask(T, feat.device)
        ht=self.ta(h.reshape(B*N,T,-1), src_mask=cm).reshape(B,N,T,-1)  # causal
        return self.head(h+ht).squeeze(-1)
astgcn=ASTGCNLite(H=GH).to(device); train_g(astgcn,"ASTGCN-lite"); eval_g(astgcn,"14. ASTGCN-lite (GCN + Causal Attn, fixed)")

# 15. GWN-lite — causal dilated convolutions: strictly left-to-right, no fix needed.
class GWNLite(nn.Module):
    def __init__(self,F=4,H=64,num_nodes=325,**k):
        super().__init__()
        self.E1=nn.Parameter(torch.randn(num_nodes,10))
        self.E2=nn.Parameter(torch.randn(10,num_nodes))
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([nn.Conv1d(H,H*2,2,dilation=2**i) for i in range(4)])
        self.gcn=DiffGCN(H,H,K=1); self.head=nn.Linear(H,1)
    def forward(self,feat,A_static):
        B,N,T,_=feat.shape
        Aa=torch.softmax(torch.relu(self.E1@self.E2),-1)
        Am=0.5*(A_static+Aa)
        h=self.proj(feat).permute(0,1,3,2).reshape(B*N,-1,T)
        skip=[]
        for cv in self.convs:
            d=cv.dilation[0]; p=(cv.kernel_size[0]-1)*d
            g=cv(F.pad(h,(p,0))); g1,g2=g.chunk(2,1)
            s=torch.tanh(g1)*torch.sigmoid(g2)
            h=h+s[:,:,:T]; skip.append(h)
        h=sum(skip).reshape(B,N,-1,T).permute(0,3,1,2).reshape(B*T,N,-1)
        h=self.gcn(h,Am).reshape(B,T,N,-1).permute(0,2,1,3)
        return self.head(h).squeeze(-1)
gwn=GWNLite(H=GH,num_nodes=NUM_NODES).to(device); train_g(gwn,"GWN-lite"); eval_g(gwn,"15. GWN-lite (Adaptive GCN + Gated TCN)")


In [ ]:
net = MaskedSTTransformerV3(NUM_NODES, A_t, node_means_t, node_stds_t,
                            hidden=HIDDEN_DIM, n_heads=N_HEADS,
                            n_layers=N_LAYERS, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
BATCH_SIZE = 2
print(f"Training ST-Transformer BIG V3-FIXED [{DATASET_NAME}] | params: {n_params/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM} layers={N_LAYERS} dropout={DROPOUT} epochs={TRAIN_EPOCHS} | "
      f"loss=Huber(beta={HUBER_BETA}) | features=5 (guarded km/h n_mean) | mask-aware spatial attn")
for ep in range(1, TRAIN_EPOCHS + 1):
    net.train()

    t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, BATCH_SIZE)
    x_list, ha_list, sin_list, cos_list, m_list, v_list = [], [], [], [], [], []

    for t0 in t0_list:
        x_list.append(speed_gpu[t0:t0+BATCH_TIME].T)
        ha_list.append(ha_prior[t0:t0+BATCH_TIME].T)
        v_list.append(valid_gpu[t0:t0+BATCH_TIME].T)
        t_idx = torch.arange(t0, t0 + BATCH_TIME, device=device)
        sin_list.append(torch.sin(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, -1).expand(NUM_NODES, -1))
        cos_list.append(torch.cos(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, -1).expand(NUM_NODES, -1))
        m_list.append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > SPARSITY).float())

    x_batch = torch.stack(x_list)
    ha_batch = torch.stack(ha_list)
    sin_batch = torch.stack(sin_list)
    cos_batch = torch.stack(cos_list)
    m_batch = torch.stack(m_list)
    v_batch = torch.stack(v_list)

    # Effective mask: invalid positions are also "unobserved" so the model never
    # ingests a fake 0-km/h reading as a real observation.
    m_eff = m_batch * v_batch

    p = net(x_batch * m_eff, m_eff, sin_batch, cos_batch, ha_batch)

    # Score loss only on (random mask hides) AND (ground truth is valid)
    loss_mask = (m_batch == 0) & (v_batch > 0)
    if not loss_mask.any():
        continue
    loss = F.smooth_l1_loss(p[loss_mask], x_batch[loss_mask], beta=HUBER_BETA)
    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
    optimizer.step(); scheduler.step()

    if ep % 100 == 0:
        a = net.last_alpha
        trust_frac = net.last_nmean_trust_frac.item() * 100
        sat0 = (a < 0.1).float().mean().item() * 100
        sat1 = (a > 0.9).float().mean().item() * 100
        print(f"Epoch {ep:4d} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.1e} | "
              f"alpha[mean={a.mean().item():.3f} max={a.max().item():.3f}] "
              f"sat<0.1: {sat0:4.1f}%  sat>0.9: {sat1:4.1f}%  nmean_trust: {trust_frac:4.1f}%")

In [ ]:
# V4 hyperparameters
HIDDEN_DIM_V4   = 128
N_LAYERS_V4     = 6
TRAIN_EPOCHS_V4 = 1500
BATCH_SIZE_V4   = 2

net_v4 = MaskedSTTransformerV4(NUM_NODES, A_t, node_means_t, node_stds_t,
                                hidden=HIDDEN_DIM_V4, n_heads=N_HEADS,
                                n_layers=N_LAYERS_V4, dropout=DROPOUT).to(device)
opt_v4 = torch.optim.Adam(net_v4.parameters(), lr=1e-3)
sch_v4 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_v4, T_max=TRAIN_EPOCHS_V4)

n_params_v4 = sum(p.numel() for p in net_v4.parameters() if p.requires_grad)
print(f"Training ST-Transformer BIG V4 [{DATASET_NAME}] | params: {n_params_v4/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM_V4} layers={N_LAYERS_V4} dropout={DROPOUT} epochs={TRAIN_EPOCHS_V4} | "
      f"features=8 (+staleness +soft_locf) | masking curriculum 60%->80%")

for ep in range(1, TRAIN_EPOCHS_V4 + 1):
    net_v4.train()
    # Curriculum: ramp masking rate from 60% to 80% over first 600 epochs
    sparsity_ep = min(SPARSITY, 0.60 + (SPARSITY - 0.60) * min(1.0, (ep - 1) / 600))

    t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, BATCH_SIZE_V4)
    x_list, ha_list, sin_list, cos_list, m_list, v_list = [], [], [], [], [], []
    for t0 in t0_list:
        x_list .append(speed_gpu[t0:t0+BATCH_TIME].T)
        ha_list.append(ha_prior[t0:t0+BATCH_TIME].T)
        v_list .append(valid_gpu[t0:t0+BATCH_TIME].T)
        t_idx  = torch.arange(t0, t0 + BATCH_TIME, device=device)
        sin_list.append(torch.sin(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY)
                        .view(1,-1).expand(NUM_NODES,-1))
        cos_list.append(torch.cos(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY)
                        .view(1,-1).expand(NUM_NODES,-1))
        m_list .append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > sparsity_ep).float())

    x_batch   = torch.stack(x_list)
    ha_batch  = torch.stack(ha_list)
    sin_batch = torch.stack(sin_list)
    cos_batch = torch.stack(cos_list)
    m_batch   = torch.stack(m_list)
    v_batch   = torch.stack(v_list)
    m_eff     = m_batch * v_batch

    p = net_v4(x_batch * m_eff, m_eff, sin_batch, cos_batch, ha_batch)

    loss_mask = (m_batch == 0) & (v_batch > 0)
    if not loss_mask.any(): continue
    loss = F.smooth_l1_loss(p[loss_mask], x_batch[loss_mask], beta=HUBER_BETA)
    opt_v4.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(net_v4.parameters(), 0.5)
    opt_v4.step(); sch_v4.step()

    if ep % 100 == 0:
        a = net_v4.last_alpha
        sat0 = (a < 0.1).float().mean().item() * 100
        sat1 = (a > 0.9).float().mean().item() * 100
        print(f"Epoch {ep:4d} | Loss: {loss.item():.4f} | LR: {sch_v4.get_last_lr()[0]:.1e} | "
              f"sparsity: {sparsity_ep:.2f} | alpha[mean={a.mean().item():.3f} "
              f"max={a.max().item():.3f}] sat<0.1: {sat0:4.1f}%  sat>0.9: {sat1:4.1f}%")


In [ ]:
net_v4.eval()
EVAL_START_V4, EVAL_LEN_V4 = 4500, 450
CHUNK_V4 = BATCH_TIME  # 48 -- same as training window

with torch.no_grad():
    x_ev4  = speed_gpu[EVAL_START_V4:EVAL_START_V4+EVAL_LEN_V4].T.unsqueeze(0)
    ha_ev4 = ha_prior [EVAL_START_V4:EVAL_START_V4+EVAL_LEN_V4].T.unsqueeze(0)
    v_ev4  = valid_gpu[EVAL_START_V4:EVAL_START_V4+EVAL_LEN_V4].T.unsqueeze(0)
    t_idx4 = torch.arange(EVAL_START_V4, EVAL_START_V4 + EVAL_LEN_V4, device=device)
    t_sin4 = (torch.sin(2*np.pi*(t_idx4%STEPS_PER_DAY)/STEPS_PER_DAY)
              .view(1,1,-1).expand(1,NUM_NODES,-1))
    t_cos4 = (torch.cos(2*np.pi*(t_idx4%STEPS_PER_DAY)/STEPS_PER_DAY)
              .view(1,1,-1).expand(1,NUM_NODES,-1))
    stds4  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    means4 = torch.tensor(node_means, device=device).view(1,-1,1)

    mae_v4_seeds, mae_ha_v4_seeds = [], []
    for seed in EVAL_SEEDS:
        m_np   = make_eval_mask_np(seed, EVAL_LEN_V4, NUM_NODES)
        m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)
        m_eff  = m_eval * v_ev4

        preds = []
        for c0 in range(0, EVAL_LEN_V4, CHUNK_V4):
            c1 = min(c0 + CHUNK_V4, EVAL_LEN_V4)
            preds.append(net_v4(x_ev4[:,:,c0:c1]*m_eff[:,:,c0:c1],
                                m_eff[:,:,c0:c1],
                                t_sin4[:,:,c0:c1], t_cos4[:,:,c0:c1],
                                ha_ev4[:,:,c0:c1]))
        p_eval = torch.cat(preds, dim=2)

        p_kmh  = (p_eval * stds4 + means4).clamp(0, 120)
        t_kmh  = (x_ev4  * stds4 + means4).clamp(0, 120)
        ha_kmh = (ha_ev4 * stds4 + means4).clamp(0, 120)
        sm     = (m_eval == 0) & (v_ev4 > 0)
        mae_v4  = torch.abs(p_kmh[sm] - t_kmh[sm]).mean().item()
        mae_ha4 = torch.abs(ha_kmh[sm] - t_kmh[sm]).mean().item()
        mae_v4_seeds .append(mae_v4)
        mae_ha_v4_seeds.append(mae_ha4)
        print(f"seed {seed}: HA={mae_ha4:.4f}  V4={mae_v4:.4f}  delta={mae_ha4-mae_v4:+.4f}")

mae_v4_seeds = np.array(mae_v4_seeds)
print(f"\nV4 MAE: {mae_v4_seeds.mean():.4f} +/- {mae_v4_seeds.std():.4f} km/h")
delta_v4 = np.array(mae_ha_v4_seeds).mean() - mae_v4_seeds.mean()
print(f"V4 vs HA: {delta_v4:+.4f} km/h  ({100*delta_v4/np.array(mae_ha_v4_seeds).mean():+.1f}%)")


In [ ]:
# V5 hyperparameters
HIDDEN_DIM_V5   = 128
N_LAYERS_V5     = 6
TRAIN_EPOCHS_V5 = 2000
BATCH_SIZE_V5   = 4

net_v5 = MaskedSTTransformerV5(NUM_NODES, A_t, node_means_t, node_stds_t,
                                hidden=HIDDEN_DIM_V5, n_heads=N_HEADS,
                                n_layers=N_LAYERS_V5, dropout=DROPOUT).to(device)
opt_v5 = torch.optim.Adam(net_v5.parameters(), lr=1e-3)
sch_v5 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_v5, T_max=TRAIN_EPOCHS_V5)

n_params_v5 = sum(p.numel() for p in net_v5.parameters() if p.requires_grad)
print(f"Training ST-Transformer BIG V5 [{DATASET_NAME}] | params: {n_params_v5/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM_V5} layers={N_LAYERS_V5} dropout={DROPOUT} epochs={TRAIN_EPOCHS_V5} | "
      f"features=9 (+2-hop n_mean) | base=soft_locf | masking curriculum 60%->80%")

for ep in range(1, TRAIN_EPOCHS_V5 + 1):
    net_v5.train()
    sparsity_ep = min(SPARSITY, 0.60 + (SPARSITY - 0.60) * min(1.0, (ep - 1) / 600))

    t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, BATCH_SIZE_V5)
    x_list, ha_list, sin_list, cos_list, m_list, v_list = [], [], [], [], [], []
    for t0 in t0_list:
        x_list .append(speed_gpu[t0:t0+BATCH_TIME].T)
        ha_list.append(ha_prior[t0:t0+BATCH_TIME].T)
        v_list .append(valid_gpu[t0:t0+BATCH_TIME].T)
        t_idx  = torch.arange(t0, t0 + BATCH_TIME, device=device)
        sin_list.append(torch.sin(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY)
                        .view(1,-1).expand(NUM_NODES,-1))
        cos_list.append(torch.cos(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY)
                        .view(1,-1).expand(NUM_NODES,-1))
        m_list .append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > sparsity_ep).float())

    x_batch   = torch.stack(x_list)
    ha_batch  = torch.stack(ha_list)
    sin_batch = torch.stack(sin_list)
    cos_batch = torch.stack(cos_list)
    m_batch   = torch.stack(m_list)
    v_batch   = torch.stack(v_list)
    m_eff     = m_batch * v_batch

    p = net_v5(x_batch * m_eff, m_eff, sin_batch, cos_batch, ha_batch)

    loss_mask = (m_batch == 0) & (v_batch > 0)
    if not loss_mask.any(): continue
    loss = F.smooth_l1_loss(p[loss_mask], x_batch[loss_mask], beta=HUBER_BETA)
    opt_v5.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(net_v5.parameters(), 0.5)
    opt_v5.step(); sch_v5.step()

    if ep % 100 == 0:
        a = net_v5.last_alpha
        sat0 = (a < 0.1).float().mean().item() * 100
        sat1 = (a > 0.9).float().mean().item() * 100
        print(f"Epoch {ep:4d} | Loss: {loss.item():.4f} | LR: {sch_v5.get_last_lr()[0]:.1e} | "
              f"sparsity: {sparsity_ep:.2f} | alpha[mean={a.mean().item():.3f} "
              f"max={a.max().item():.3f}] sat<0.1: {sat0:4.1f}%  sat>0.9: {sat1:4.1f}%")


In [ ]:
net_v5.eval()
EVAL_START_V5, EVAL_LEN_V5 = 4500, 450
CHUNK_V5 = BATCH_TIME  # 48 -- same as training window

with torch.no_grad():
    x_ev5  = speed_gpu[EVAL_START_V5:EVAL_START_V5+EVAL_LEN_V5].T.unsqueeze(0)
    ha_ev5 = ha_prior [EVAL_START_V5:EVAL_START_V5+EVAL_LEN_V5].T.unsqueeze(0)
    v_ev5  = valid_gpu[EVAL_START_V5:EVAL_START_V5+EVAL_LEN_V5].T.unsqueeze(0)
    t_idx5 = torch.arange(EVAL_START_V5, EVAL_START_V5 + EVAL_LEN_V5, device=device)
    t_sin5 = (torch.sin(2*np.pi*(t_idx5%STEPS_PER_DAY)/STEPS_PER_DAY)
              .view(1,1,-1).expand(1,NUM_NODES,-1))
    t_cos5 = (torch.cos(2*np.pi*(t_idx5%STEPS_PER_DAY)/STEPS_PER_DAY)
              .view(1,1,-1).expand(1,NUM_NODES,-1))
    stds5  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    means5 = torch.tensor(node_means, device=device).view(1,-1,1)

    mae_v5_seeds, mae_ha_v5_seeds = [], []
    for seed in EVAL_SEEDS:
        m_np   = make_eval_mask_np(seed, EVAL_LEN_V5, NUM_NODES)
        m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)
        m_eff  = m_eval * v_ev5

        preds = []
        for c0 in range(0, EVAL_LEN_V5, CHUNK_V5):
            c1 = min(c0 + CHUNK_V5, EVAL_LEN_V5)
            preds.append(net_v5(x_ev5[:,:,c0:c1]*m_eff[:,:,c0:c1],
                                m_eff[:,:,c0:c1],
                                t_sin5[:,:,c0:c1], t_cos5[:,:,c0:c1],
                                ha_ev5[:,:,c0:c1]))
        p_eval = torch.cat(preds, dim=2)

        p_kmh  = (p_eval * stds5 + means5).clamp(0, 120)
        t_kmh  = (x_ev5  * stds5 + means5).clamp(0, 120)
        ha_kmh = (ha_ev5 * stds5 + means5).clamp(0, 120)
        sm     = (m_eval == 0) & (v_ev5 > 0)
        mae_v5  = torch.abs(p_kmh[sm] - t_kmh[sm]).mean().item()
        mae_ha5 = torch.abs(ha_kmh[sm] - t_kmh[sm]).mean().item()
        mae_v5_seeds .append(mae_v5)
        mae_ha_v5_seeds.append(mae_ha5)
        print(f"seed {seed}: HA={mae_ha5:.4f}  V5={mae_v5:.4f}  delta={mae_ha5-mae_v5:+.4f}")

mae_v5_seeds = np.array(mae_v5_seeds)
print(f"\nV5 MAE: {mae_v5_seeds.mean():.4f} +/- {mae_v5_seeds.std():.4f} km/h")
delta_v5 = np.array(mae_ha_v5_seeds).mean() - mae_v5_seeds.mean()
print(f"V5 vs HA: {delta_v5:+.4f} km/h  ({100*delta_v5/np.array(mae_ha_v5_seeds).mean():+.1f}%)")


In [ ]:
net.eval()
EVAL_START, EVAL_LEN = 4500, 450
# ▶ FIX: chunk size matches training window — LOCF context is consistent with training.
# Original CHUNK = BATCH_TIME*4 = 192 gave the LOCF anchor 4× more history than
# the model was trained with, artificially boosting its quality.
CHUNK = BATCH_TIME  # = 48, same as training

with torch.no_grad():
    x_eval = speed_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ha_eval = ha_prior[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    v_eval = valid_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    t_idx = torch.arange(EVAL_START, EVAL_START + EVAL_LEN, device=device)
    t_sin = torch.sin(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, 1, -1).expand(1, NUM_NODES, -1)
    t_cos = torch.cos(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, 1, -1).expand(1, NUM_NODES, -1)

    stds  = torch.tensor(node_stds,  device=device).view(1, -1, 1)
    means = torch.tensor(node_means, device=device).view(1, -1, 1)

    mae_mod_seeds, mae_ha_seeds, scored_frac_seeds = [], [], []

    for seed in EVAL_SEEDS:
        # ▶ FIX: identical mask to all other models via shared numpy RNG
        m_np = make_eval_mask_np(seed, EVAL_LEN, NUM_NODES)  # [T,N]
        m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)  # [1,N,T]
        m_eff  = m_eval * v_eval

        preds = []
        for c0 in range(0, EVAL_LEN, CHUNK):
            c1 = min(c0 + CHUNK, EVAL_LEN)
            p_c = net(x_eval[:, :, c0:c1] * m_eff[:, :, c0:c1],
                      m_eff[:, :, c0:c1],
                      t_sin[:, :, c0:c1], t_cos[:, :, c0:c1],
                      ha_eval[:, :, c0:c1])
            preds.append(p_c)
        p_eval = torch.cat(preds, dim=2)

        p_kmh  = (p_eval * stds + means).clamp(0, 120)
        t_kmh  = (x_eval * stds + means).clamp(0, 120)
        ha_kmh = (ha_eval * stds + means).clamp(0, 120)

        # Score only on (random mask hides) AND (ground truth is real)
        score_mask = (m_eval == 0) & (v_eval > 0)
        n_scored   = int(score_mask.sum().item())
        n_held     = int((m_eval == 0).sum().item())
        scored_frac = n_scored / max(n_held, 1)

        mae_mod = torch.abs(p_kmh[score_mask] - t_kmh[score_mask]).mean().item()
        mae_ha  = torch.abs(ha_kmh[score_mask] - t_kmh[score_mask]).mean().item()
        mae_mod_seeds.append(mae_mod)
        mae_ha_seeds.append(mae_ha)
        scored_frac_seeds.append(scored_frac)
        print(f"seed {seed}: HA={mae_ha:.4f}  TFv3fix={mae_mod:.4f}  delta={mae_ha-mae_mod:+.4f}  "
              f"scored={n_scored}/{n_held} ({100*scored_frac:.1f}% valid)")

mae_mod_seeds = np.array(mae_mod_seeds)
mae_ha_seeds  = np.array(mae_ha_seeds)

print(f"\nFINAL RESULTS (80% Sparsity, ST-Transformer BIG V3-FIXED [{DATASET_NAME}], {len(EVAL_SEEDS)} seeds)")
print(f"Historical Average MAE: {mae_ha_seeds.mean():.4f} +/- {mae_ha_seeds.std():.4f} km/h")
print(f"V3-FIXED MAE:           {mae_mod_seeds.mean():.4f} +/- {mae_mod_seeds.std():.4f} km/h")
delta = mae_ha_seeds.mean() - mae_mod_seeds.mean()
print(f"Delta vs HA:            {delta:+.4f} km/h ({100*delta/mae_ha_seeds.mean():+.1f}%)")
print(f"Mean scored fraction:   {100*np.mean(scored_frac_seeds):.1f}%")


In [ ]:
# Final comparison table
RESULTS["16. MaskedSTTransformerV3-FIXED"] = mae_mod_seeds
RESULTS["17. MaskedSTTransformerV4"] = mae_v4_seeds
RESULTS["18. MaskedSTTransformerV5 (ours)"] = mae_v5_seeds

ha_mae = RESULTS.get("1. Historical Average (HA)", mae_ha_seeds).mean()
print()
print("="*72)
print(f"{'PEMS-BAY IMPUTATION BENCHMARK  (80% Sparsity, 5 seeds)':^72}")
print("="*72)
print(f"{'Model':<52} {'MAE':>8}  {'Std':>6}  {'vs HA':>7}")
print("-"*72)
for name, arr in sorted(RESULTS.items(), key=lambda kv: kv[1].mean()):
    tag = " < OURS" if name == "18. MaskedSTTransformerV5 (ours)" else ""
    diff = arr.mean() - ha_mae
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>6.4f}  {diff:>+7.4f}{tag}")
print("-"*72)
ours_v5 = RESULTS["18. MaskedSTTransformerV5 (ours)"].mean()
delta = ha_mae - ours_v5
print(f"Ours (V5) vs HA: {delta:+.4f} km/h  ({100*delta/ha_mae:+.1f}%)")
print("="*72)


In [ ]:
# ── Extended Evaluation Metrics ─────────────────────────────────────────────────────────────────
# Computes RMSE, MAPE, R², MBE, MedAE, Pearson r, Hit@5 km/h, Hit@10 km/h
# for HA, LOCF, V3, V4, V5 using 2 seeds (42-43). Results stored in EXT_METRICS + raw arrays.

import numpy as np, torch

_EXT_SEEDS = [42, 43]

def _run_model_ext(model, eval_start=4500, eval_len=450):
    chunk  = BATCH_TIME
    model.eval()
    x_ev   = speed_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    ha_ev  = ha_prior [eval_start:eval_start+eval_len].T.unsqueeze(0)
    v_ev   = valid_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    t_idx  = torch.arange(eval_start, eval_start+eval_len, device=device)
    t_s    = torch.sin(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    t_c    = torch.cos(2*np.pi*(t_idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    stds_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    means_ = torch.tensor(node_means, device=device).view(1,-1,1)
    all_p, all_t = [], []
    with torch.no_grad():
        for seed in _EXT_SEEDS:
            m_np   = make_eval_mask_np(seed, eval_len, NUM_NODES)
            m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff  = m_eval * v_ev
            preds  = []
            for c0 in range(0, eval_len, chunk):
                c1 = min(c0+chunk, eval_len)
                preds.append(model(x_ev[:,:,c0:c1]*m_eff[:,:,c0:c1], m_eff[:,:,c0:c1],
                                   t_s[:,:,c0:c1], t_c[:,:,c0:c1], ha_ev[:,:,c0:c1]))
            p  = torch.cat(preds, dim=2)
            pk = (p   * stds_ + means_).clamp(0, 120)
            tk = (x_ev * stds_ + means_).clamp(0, 120)
            sm = (m_eval == 0) & (v_ev > 0)
            all_p.append(pk[sm].cpu().numpy())
            all_t.append(tk[sm].cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

def _run_locf_ext(eval_start=4500, eval_len=450):
    x_ev   = speed_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    ha_ev  = ha_prior [eval_start:eval_start+eval_len].T.unsqueeze(0)
    v_ev   = valid_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    stds_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    means_ = torch.tensor(node_means, device=device).view(1,-1,1)
    all_p, all_t = [], []
    with torch.no_grad():
        for seed in _EXT_SEEDS:
            m_np   = make_eval_mask_np(seed, eval_len, NUM_NODES)
            m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff  = m_eval * v_ev
            prev   = ha_ev[:,:,0:1].clone()
            out    = []
            for t in range(eval_len):
                obs  = x_ev[:,:,t:t+1] * m_eff[:,:,t:t+1]
                prev = torch.where(m_eff[:,:,t:t+1] > 0, obs, prev)
                out.append(prev.clone())
            p  = torch.cat(out, dim=2)
            pk = (p   * stds_ + means_).clamp(0, 120)
            tk = (x_ev * stds_ + means_).clamp(0, 120)
            sm = (m_eval == 0) & (v_ev > 0)
            all_p.append(pk[sm].cpu().numpy())
            all_t.append(tk[sm].cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

def _run_ha_ext(eval_start=4500, eval_len=450):
    x_ev   = speed_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    ha_ev  = ha_prior [eval_start:eval_start+eval_len].T.unsqueeze(0)
    v_ev   = valid_gpu[eval_start:eval_start+eval_len].T.unsqueeze(0)
    stds_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    means_ = torch.tensor(node_means, device=device).view(1,-1,1)
    all_p, all_t = [], []
    with torch.no_grad():
        for seed in _EXT_SEEDS:
            m_np   = make_eval_mask_np(seed, eval_len, NUM_NODES)
            m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)
            pk = (ha_ev * stds_ + means_).clamp(0, 120)
            tk = (x_ev  * stds_ + means_).clamp(0, 120)
            sm = (m_eval == 0) & (v_ev > 0)
            all_p.append(pk[sm].cpu().numpy())
            all_t.append(tk[sm].cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

def extended_metrics(pred, true):
    err   = pred - true
    mae   = np.abs(err).mean()
    rmse  = np.sqrt((err**2).mean())
    mape  = (np.abs(err) / np.maximum(true, 1.0)).mean() * 100
    ss_r  = (err**2).sum();  ss_t = ((true - true.mean())**2).sum()
    r2    = 1 - ss_r / ss_t
    mbe   = err.mean()
    medae = np.median(np.abs(err))
    pear  = float(np.corrcoef(pred, true)[0, 1])
    hit5  = (np.abs(err) <  5.0).mean() * 100
    hit10 = (np.abs(err) < 10.0).mean() * 100
    return dict(MAE=mae, RMSE=rmse, MedAE=medae, MAPE=mape,
                R2=r2, Pearson=pear, MBE=mbe, Hit5=hit5, Hit10=hit10)

print("Computing extended metrics (seeds 42-43) ...")
p_ha, t_ha = _run_ha_ext()
p_lc, t_lc = _run_locf_ext()
p_v3, t_v3 = _run_model_ext(net)
p_v4, t_v4 = _run_model_ext(net_v4)
p_v5, t_v5 = _run_model_ext(net_v5)

m_ha = extended_metrics(p_ha, t_ha)
m_lc = extended_metrics(p_lc, t_lc)
m_v3 = extended_metrics(p_v3, t_v3)
m_v4 = extended_metrics(p_v4, t_v4)
m_v5 = extended_metrics(p_v5, t_v5)

EXT_METRICS = {
    "1. Hist. Average": m_ha,
    "2. LOCF":          m_lc,
    "3. V3-FIXED":      m_v3,
    "4. V4":            m_v4,
    "5. V5 (ours)":     m_v5,
}

cols_ext = ["MAE", "RMSE", "MedAE", "MAPE%", "R2", "Pearson", "MBE", "Hit@5%", "Hit@10%"]
w = 11
W = 20 + w * len(cols_ext)
print()
print("=" * W)
print(f"{'EXTENDED EVALUATION METRICS  --  PEMS-BAY  (80% sparsity, seeds 42-43)':^{W}}")
print("=" * W)
print(f"{'Model':<20}" + "".join(f"{c:>{w}}" for c in cols_ext))
print("-" * W)
for name, m in EXT_METRICS.items():
    row = (f"{name:<20}"
           f"{m['MAE']:>{w}.4f}"
           f"{m['RMSE']:>{w}.4f}"
           f"{m['MedAE']:>{w}.4f}"
           f"{m['MAPE']:>{w}.2f}"
           f"{m['R2']:>{w}.4f}"
           f"{m['Pearson']:>{w}.4f}"
           f"{m['MBE']:>{w}.4f}"
           f"{m['Hit5']:>{w}.2f}"
           f"{m['Hit10']:>{w}.2f}")
    tag = "  <-- BEST" if name == "5. V5 (ours)" else ""
    print(row + tag)
print("=" * W)


## Visualisations

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Model metadata ────────────────────────────────────────────────────
FAMILY_COLOR = {
    'stat':     '#9E9E9E',
    'linear':   '#78909C',
    'nonparam': '#90A4AE',
    'nn':       '#42A5F5',
    'rnn':      '#29B6F6',
    'conv':     '#26C6DA',
    'attn':     '#66BB6A',
    'graph':    '#FFA726',
    'ours':     '#B71C1C',
}
FAMILY_LABEL = {
    'stat':'Statistical', 'linear':'Linear', 'nonparam':'Non-parametric',
    'nn':'Neural (per-node)', 'rnn':'RNN', 'conv':'TCN',
    'attn':'Attention', 'graph':'Graph', 'ours':'Ours',
}
MODEL_FAMILY = {
    '1. Historical Average (HA)':           'stat',
    '2. LOCF (Last-Obs Carried Forward)':   'stat',
    '3. Global Mean (per-node train mean)':  'stat',
    '4. Node-wise Ridge Regression':         'linear',
    '5. KNN Imputer (k=5, masked-dist)':     'nonparam',
    '6.  MLP (per-node + node-emb)':         'nn',
    '7.  LSTM (per-node)':                   'rnn',
    '8.  BiLSTM->2L-LSTM (causal, fixed)':   'rnn',
    '9.  GRU (per-node)':                    'rnn',
    '10. TCN (causal dilated, per-node)':    'conv',
    '11. SAITS-lite (causal Attn + node-emb, fixed)': 'attn',
    '12. BRITS-lite (forward GRU only, fixed)': 'rnn',
    '13. DCRNN-lite (DiffGCN + GRU)':        'graph',
    '14. ASTGCN-lite (GCN + Causal Attn, fixed)': 'graph',
    '15. GWN-lite (Adaptive GCN + Gated TCN)': 'graph',
    '16. MaskedSTTransformerV3-FIXED':       'ours',
    '17. MaskedSTTransformerV4':             'ours',
    '18. MaskedSTTransformerV5 (ours)':      'ours',
}
SHORT = {
    '1. Historical Average (HA)':           'Hist. Avg.',
    '2. LOCF (Last-Obs Carried Forward)':   'LOCF',
    '3. Global Mean (per-node train mean)':  'Global Mean',
    '4. Node-wise Ridge Regression':         'Ridge',
    '5. KNN Imputer (k=5, masked-dist)':     'KNN (k=5)',
    '6.  MLP (per-node + node-emb)':         'MLP',
    '7.  LSTM (per-node)':                   'LSTM',
    '8.  BiLSTM->2L-LSTM (causal, fixed)':   'BiLSTM→LSTM',
    '9.  GRU (per-node)':                    'GRU',
    '10. TCN (causal dilated, per-node)':    'TCN',
    '11. SAITS-lite (causal Attn + node-emb, fixed)': 'SAITS-lite',
    '12. BRITS-lite (forward GRU only, fixed)': 'BRITS-lite',
    '13. DCRNN-lite (DiffGCN + GRU)':        'DCRNN-lite',
    '14. ASTGCN-lite (GCN + Causal Attn, fixed)': 'ASTGCN-lite',
    '15. GWN-lite (Adaptive GCN + Gated TCN)': 'GWN-lite',
    '16. MaskedSTTransformerV3-FIXED':       'MST-V3',
    '17. MaskedSTTransformerV4':             'MST-V4',
    '18. MaskedSTTransformerV5 (ours)':      'MST-V5 ★',
}

ha_mae  = RESULTS['1. Historical Average (HA)'].mean()
locf_mae = RESULTS['2. LOCF (Last-Obs Carried Forward)'].mean()

# Sort worst → best so best appears at top of horizontal bar chart
items  = sorted(RESULTS.items(), key=lambda kv: kv[1].mean(), reverse=True)
labels = [SHORT.get(k, k) for k, _ in items]
maes   = np.array([v.mean() for _, v in items])
stds   = np.array([v.std()  for _, v in items])
colors = [FAMILY_COLOR[MODEL_FAMILY.get(k, 'stat')] for k, _ in items]

# ════════════════════════════════════════════════════════════════════════
# Figure 1 — Full benchmark comparison
# ════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(labels))
ax.barh(y, maes, xerr=stds, color=colors, height=0.65,
        error_kw=dict(ecolor='#333', capsize=3, elinewidth=1.2))
ax.axvline(ha_mae,   color='#FF7043', ls='--', lw=1.5, alpha=0.8, label=f'HA ({ha_mae:.2f})')
ax.axvline(locf_mae, color='#78909C', ls=':',  lw=1.5, alpha=0.8, label=f'LOCF ({locf_mae:.2f})')
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('MAE (km/h)')
ax.set_title('PEMS-BAY Imputation Benchmark  |  80% Sparsity, 5 Seeds',
             fontsize=12, fontweight='bold')
ax.invert_yaxis()

# Family legend
seen_fams = dict.fromkeys(MODEL_FAMILY.get(k, 'stat') for k, _ in items)
handles = [mpatches.Patch(color=FAMILY_COLOR[f], label=FAMILY_LABEL[f])
           for f in seen_fams]
leg1 = ax.legend(handles=handles, loc='lower right', fontsize=8,
                 title='Model family', title_fontsize=8, framealpha=0.9)
ax.add_artist(leg1)
ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
ax.grid(axis='x', alpha=0.25)
ax.set_xlim(0, max(maes) * 1.13)

# Annotate our best model
best_k = '18. MaskedSTTransformerV5 (ours)'
if best_k in RESULTS:
    bm = RESULTS[best_k].mean()
    bi = labels.index(SHORT[best_k])
    ax.annotate(f'+{100*(ha_mae-bm)/ha_mae:.1f}% vs HA',
                xy=(bm, bi), xytext=(bm+0.08, bi-0.6),
                fontsize=8.5, color='#B71C1C', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.3))

plt.tight_layout()
plt.savefig('fig1_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_benchmark.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 2 — Three-panel diagnostics
# ════════════════════════════════════════════════════════════════════════

# -- 2a: Collect alpha values on eval data (2 seeds for speed) -----------
def _collect_alpha(model, n_seeds=2):
    model.eval()
    alphas = []
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1 = min(c0+BATCH_TIME, EL)
                _  = model(x_e[:,:,c0:c1]*m_eff[:,:,c0:c1], m_eff[:,:,c0:c1],
                           ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                alphas.append(model.last_alpha.cpu().float().flatten().numpy())
    return np.concatenate(alphas)

# -- 2b: Collect staleness-binned MAE for V4 and V5 ----------------------
def _staleness_mae(model, n_seeds=2):
    """Return (staleness_flat, abs_error_flat) for held-out positions."""
    model.eval()
    stale_all, err_all = [], []
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn_  = torch.tensor(node_means, device=device).view(1,-1,1)
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1  = min(c0+BATCH_TIME, EL)
                xc  = x_e[:,:,c0:c1]; mc = m_eff[:,:,c0:c1]
                out = model(xc*mc, mc, ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                # Collect staleness from the model's last run
                _, stale_c, _ = model._compute_causal_signals(xc*mc, mc, ha_e[:,:,c0:c1])
                sm  = (m_ev[:,:,c0:c1] == 0) & (v_e[:,:,c0:c1] > 0)
                if not sm.any(): continue
                p_km = (out * st_ + mn_).clamp(0, 120)
                t_km = (xc  * st_ + mn_).clamp(0, 120)
                stale_all.append(stale_c[sm].cpu().numpy())
                err_all  .append(torch.abs(p_km[sm] - t_km[sm]).cpu().numpy())
    return np.concatenate(stale_all), np.concatenate(err_all)

print('Collecting alpha & staleness diagnostics...')
alpha_v3 = _collect_alpha(net)
alpha_v4 = _collect_alpha(net_v4)
alpha_v5 = _collect_alpha(net_v5)
stale_v4, err_v4 = _staleness_mae(net_v4)
stale_v5, err_v5 = _staleness_mae(net_v5)
print('Done.')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel A: V3 → V4 → V5 MAE progression ───────────────────────────
ax = axes[0]
v_keys   = ['16. MaskedSTTransformerV3-FIXED',
            '17. MaskedSTTransformerV4',
            '18. MaskedSTTransformerV5 (ours)']
v_labels = ['V3', 'V4', 'V5']
v_maes   = [RESULTS[k].mean() for k in v_keys]
v_stds   = [RESULTS[k].std()  for k in v_keys]
v_cols   = ['#EF5350', '#E53935', '#B71C1C']
bars = ax.bar(v_labels, v_maes, yerr=v_stds, color=v_cols, width=0.5,
              capsize=6, error_kw=dict(elinewidth=1.8), zorder=3)
ax.axhline(locf_mae, color='#78909C', ls='--', lw=1.2, label=f'LOCF ({locf_mae:.3f})')
ax.axhline(ha_mae,   color='#FF7043', ls=':',  lw=1.2, label=f'HA ({ha_mae:.3f})')
for bar, m, s in zip(bars, v_maes, v_stds):
    ax.text(bar.get_x()+bar.get_width()/2, m+s+0.008,
            f'{m:.4f}', ha='center', fontsize=9, fontweight='bold')
# Improvement arrows between bars
for i in range(len(v_maes)-1):
    delta = v_maes[i] - v_maes[i+1]
    ax.annotate('', xy=(i+1, v_maes[i+1]+0.02), xytext=(i, v_maes[i]+0.02),
                arrowprops=dict(arrowstyle='->', color='#444', lw=1.2))
    ax.text((i+0.5), max(v_maes[i], v_maes[i+1])+0.04,
            f'-{delta*1000:.0f}m', ha='center', fontsize=8, color='#444')
ax.set_ylabel('MAE (km/h)'); ax.set_title('Our Model: V3 → V4 → V5', fontweight='bold')
ax.set_ylim(0, max(v_maes)*1.4); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25, zorder=0)

# ── Panel B: Per-seed stability for top models ────────────────────────
ax = axes[1]
top_keys = [
    ('18. MaskedSTTransformerV5 (ours)', 'MST-V5', '#B71C1C'),
    ('17. MaskedSTTransformerV4',        'MST-V4', '#E53935'),
    ('16. MaskedSTTransformerV3-FIXED',  'MST-V3', '#EF5350'),
    ('2. LOCF (Last-Obs Carried Forward)','LOCF',  '#78909C'),
    ('12. BRITS-lite (forward GRU only, fixed)','BRITS','#29B6F6'),
    ('10. TCN (causal dilated, per-node)','TCN',   '#26C6DA'),
    ('1. Historical Average (HA)',        'HA',    '#FF7043'),
]
rng_jitter = np.random.default_rng(0)
for i, (k, lbl, col) in enumerate(top_keys):
    if k not in RESULTS: continue
    arr = RESULTS[k]
    jitter = rng_jitter.uniform(-0.15, 0.15, len(arr))
    ax.scatter(np.full(len(arr), i)+jitter, arr, color=col, s=45, zorder=4, alpha=0.85)
    ax.hlines(arr.mean(), i-0.3, i+0.3, colors=col, linewidth=2.5, zorder=5)
ax.set_xticks(range(len(top_keys)))
ax.set_xticklabels([l for _, l, _ in top_keys], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('MAE (km/h)'); ax.set_title('Per-Seed Stability (Top Models)', fontweight='bold')
ax.grid(axis='y', alpha=0.25)

# ── Panel C: Alpha gate distribution V3 / V4 / V5 ────────────────────
ax = axes[2]
bins = np.linspace(0, 1, 45)
for alpha, lbl, col in [(alpha_v3,'V3','#EF5350'),
                         (alpha_v4,'V4','#E53935'),
                         (alpha_v5,'V5','#B71C1C')]:
    ax.hist(alpha, bins=bins, alpha=0.55, color=col, density=True,
            histtype='stepfilled', label=f'{lbl}  mean={alpha.mean():.3f}')
    ax.axvline(alpha.mean(), color=col, lw=1.5, ls='--')
ax.set_xlabel('alpha  (gate weight on residual correction)')
ax.set_ylabel('Density')
ax.set_title('Meta-Gate Alpha Distribution\n(eval, 2 seeds)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2)
ax.text(0.97, 0.97, 'alpha~0: rely on LOCF/soft_locf\nalpha~1: full correction',
        transform=ax.transAxes, ha='right', va='top', fontsize=7.5,
        style='italic', color='#555')

plt.suptitle('PEMS-BAY 80% Sparsity — Diagnostic Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig2_diagnostics.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 3 — MAE vs staleness: does V5 handle stale positions better?
# ════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

BINS = [0, 0.05, 0.15, 0.30, 0.50, 0.75, 1.01]
BIN_LABELS = ['0-2', '2-7', '7-14', '14-24', '24-36', '36-48']

for ax, (stale, err, lbl, col) in zip(
        axes,
        [(stale_v4, err_v4, 'V4', '#E53935'),
         (stale_v5, err_v5, 'V5', '#B71C1C')]):
    bin_maes, bin_counts = [], []
    for lo, hi in zip(BINS[:-1], BINS[1:]):
        mask = (stale >= lo) & (stale < hi)
        bin_maes  .append(err[mask].mean() if mask.any() else np.nan)
        bin_counts.append(mask.sum())
    x = np.arange(len(BIN_LABELS))
    bars = ax.bar(x, bin_maes, color=col, alpha=0.85, width=0.6, zorder=3)
    ax2  = ax.twinx()
    ax2.plot(x, [c/1e3 for c in bin_counts], 'o--', color='#555', lw=1.2,
             ms=5, label='Sample count (k)')
    ax2.set_ylabel('Sample count (thousands)', fontsize=9)
    ax2.legend(fontsize=8, loc='upper right')
    ax.set_xticks(x); ax.set_xticklabels([f'{b}\nsteps' for b in BIN_LABELS], fontsize=9)
    ax.set_xlabel('Steps since last observation')
    ax.set_ylabel('Mean Abs Error (km/h)')
    ax.set_title(f'{lbl}: MAE by LOCF staleness', fontweight='bold')
    ax.grid(axis='y', alpha=0.25, zorder=0)
    for bar, m in zip(bars, bin_maes):
        if not np.isnan(m):
            ax.text(bar.get_x()+bar.get_width()/2, m+0.01, f'{m:.2f}',
                    ha='center', va='bottom', fontsize=8)

plt.suptitle('MAE vs LOCF Staleness  |  PEMS-BAY 80% Sparsity',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_staleness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_staleness.png')


# ════════════════════════════════════════════════════════════════════════
# Extended helpers for Figs 4-9
# ════════════════════════════════════════════════════════════════════════

def _collect_alpha_stale(model, n_seeds=2):
    """Return (alpha, staleness) aligned per held-out position."""
    model.eval()
    al, st = [], []
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1 = min(c0+BATCH_TIME, EL)
                xc = x_e[:,:,c0:c1]; mc = m_eff[:,:,c0:c1]
                _  = model(xc*mc, mc, ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                _, stale_c, _ = model._compute_causal_signals(xc*mc, mc, ha_e[:,:,c0:c1])
                sm = (m_ev[:,:,c0:c1]==0) & (v_e[:,:,c0:c1]>0)
                if not sm.any(): continue
                al.append(model.last_alpha[sm].cpu().numpy())
                st.append(stale_c[sm].cpu().numpy())
    return np.concatenate(al), np.concatenate(st)

def _collect_tod_errors(model, n_seeds=2):
    """Return (abs_err_kmh, hour_of_day) per held-out position."""
    model.eval()
    errs, hours = [], []
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn_  = torch.tensor(node_means, device=device).view(1,-1,1)
    tod_hr = ((ti % STEPS_PER_DAY).float() / (STEPS_PER_DAY / 24)).cpu().numpy()
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1  = min(c0+BATCH_TIME, EL)
                xc  = x_e[:,:,c0:c1]; mc = m_eff[:,:,c0:c1]
                pred = model(xc*mc, mc, ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                p_km = (pred * st_ + mn_).clamp(0, 120)
                t_km = (xc   * st_ + mn_).clamp(0, 120)
                sm   = (m_ev[:,:,c0:c1]==0) & (v_e[:,:,c0:c1]>0)
                if not sm.any(): continue
                hr_c = torch.tensor(tod_hr[c0:c1], device=device).view(1,1,-1).expand_as(xc)
                errs .append(torch.abs(p_km - t_km)[sm].cpu().numpy())
                hours.append(hr_c[sm].cpu().numpy())
    return np.concatenate(errs), np.concatenate(hours)

def _per_node_mae(model, n_seeds=2):
    """Return [NUM_NODES] per-node MAE in km/h."""
    model.eval()
    sums = torch.zeros(NUM_NODES, device=device)
    cnts = torch.zeros(NUM_NODES, device=device)
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn_  = torch.tensor(node_means, device=device).view(1,-1,1)
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1  = min(c0+BATCH_TIME, EL)
                xc  = x_e[:,:,c0:c1]; mc = m_eff[:,:,c0:c1]
                pred = model(xc*mc, mc, ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                p_km = (pred * st_ + mn_).clamp(0, 120)
                t_km = (xc   * st_ + mn_).clamp(0, 120)
                sm   = (m_ev[:,:,c0:c1]==0) & (v_e[:,:,c0:c1]>0)
                sums += (torch.abs(p_km - t_km)[0] * sm[0].float()).sum(dim=1)
                cnts += sm[0].float().sum(dim=1)
    return (sums / (cnts + 1e-8)).cpu().numpy()

def _locf_per_node_mae(n_seeds=2):
    """Return [NUM_NODES] per-node LOCF MAE in km/h."""
    ES, EL = 4500, 450
    x_np = speed_norm[ES:ES+EL]; v_np = valid_raw[ES:ES+EL]
    sums = np.zeros(NUM_NODES); cnts = np.zeros(NUM_NODES)
    for seed in EVAL_SEEDS[:n_seeds]:
        m_np = make_eval_mask_np(seed, EL, NUM_NODES)
        lf   = locf(x_np.T, v_np.T, m_np.T, np.arange(ES, ES+EL))  # [N,T] km/h
        gt   = (x_np.T * node_stds[:,None] + node_means[:,None]).clip(0, 120)
        sm   = (m_np.T == 0) & (v_np.T > 0)  # [N,T]
        sums += (np.abs(lf - gt) * sm).sum(axis=1)
        cnts += sm.sum(axis=1)
    return sums / (cnts + 1e-8)

print('Collecting data for Figs 4-9...')
al_v4, st_v4 = _collect_alpha_stale(net_v4)
al_v5, st_v5 = _collect_alpha_stale(net_v5)
tod_err_v5, tod_hr_v5 = _collect_tod_errors(net_v5)
tod_err_locf, tod_hr_locf = _collect_tod_errors.__wrapped__(net_v5) if False else (None, None)
node_mae_v5   = _per_node_mae(net_v5)
node_mae_locf = _locf_per_node_mae()
# LOCF per-position errors for density plot
ES2, EL2 = 4500, 450
_xnp = speed_norm[ES2:ES2+EL2]; _vnp = valid_raw[ES2:ES2+EL2]
_mnp = make_eval_mask_np(42, EL2, NUM_NODES)
_lf  = locf(_xnp.T, _vnp.T, _mnp.T, np.arange(ES2, ES2+EL2))
_gt  = (_xnp.T * node_stds[:,None] + node_means[:,None]).clip(0, 120)
_sm  = (_mnp.T == 0) & (_vnp.T > 0)
locf_err_flat = np.abs(_lf - _gt)[_sm]
# LOCF TOD errors
_tod_hr_np = (np.arange(ES2, ES2+EL2) % STEPS_PER_DAY) / (STEPS_PER_DAY / 24)
tod_hr_locf  = np.tile(_tod_hr_np, NUM_NODES).reshape(NUM_NODES, EL2)[_sm]
tod_err_locf = locf_err_flat
print('Done.')

# ════════════════════════════════════════════════════════════════════════
# Figure 4 — Per-position error density (V4, V5, LOCF)
# ════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
xlim = 12
bins = np.linspace(0, xlim, 80)
for arr, lbl, col in [
        (locf_err_flat, 'LOCF',  '#78909C'),
        (err_v4,        'V4',    '#E53935'),
        (err_v5,        'V5',    '#B71C1C')]:
    ax.hist(arr.clip(0, xlim), bins=bins, density=True, histtype='stepfilled',
            alpha=0.45, color=col, label=f'{lbl}  mean={arr.mean():.2f}')
    ax.axvline(arr.mean(), color=col, lw=1.8, ls='--')
ax.set_xlabel('Absolute error (km/h)'); ax.set_ylabel('Density')
ax.set_title('Per-Position Error Distribution', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2)

ax = axes[1]
# CDF
for arr, lbl, col in [
        (locf_err_flat, 'LOCF',  '#78909C'),
        (err_v4,        'V4',    '#E53935'),
        (err_v5,        'V5',    '#B71C1C')]:
    xs = np.sort(arr)
    ys = np.arange(1, len(xs)+1) / len(xs)
    # subsample for speed
    step = max(1, len(xs)//2000)
    ax.plot(xs[::step], ys[::step], color=col, lw=2, label=lbl)
    for p, ls in [(0.50,'--'),(0.75,':'),(0.90,'-.')]:  # percentile markers
        pv = np.percentile(arr, p*100)
        ax.axvline(pv, color=col, lw=0.8, ls=ls, alpha=0.6)
ax.set_xlim(0, 10); ax.set_xlabel('Absolute error (km/h)')
ax.set_ylabel('Fraction of positions'); ax.set_title('Error CDF', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2)
ax.text(0.97,0.05,'dashes: P50 / P75 / P90',transform=ax.transAxes,
        ha='right',fontsize=7.5,style='italic',color='#555')

plt.suptitle('Error Density & CDF  |  PEMS-BAY 80% Sparsity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_error_density.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig4_error_density.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 5 — Alpha x Staleness 2D density (V4 vs V5)
# ════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (al, st, lbl, col) in zip(axes, [
        (al_v4, st_v4, 'V4', 'Reds'),
        (al_v5, st_v5, 'V5', 'RdPu')]):
    hb = ax.hexbin(st, al, gridsize=40, cmap=col, mincnt=1,
                   extent=[0, 1, 0, 1], bins='log')
    plt.colorbar(hb, ax=ax, label='log(count)')
    # Trend line (mean alpha per staleness bin)
    s_bins = np.linspace(0, 1, 20)
    trend_x, trend_y = [], []
    for lo, hi in zip(s_bins[:-1], s_bins[1:]):
        mask = (st >= lo) & (st < hi)
        if mask.sum() > 100:
            trend_x.append((lo+hi)/2)
            trend_y.append(al[mask].mean())
    ax.plot(trend_x, trend_y, 'w-o', ms=4, lw=2, label='mean alpha')
    ax.set_xlabel('Staleness (steps since last obs / 48)')
    ax.set_ylabel('Alpha (gate weight)')
    ax.set_title(f'{lbl}: Alpha vs Staleness', fontweight='bold')
    ax.legend(fontsize=8)
    corr = np.corrcoef(st, al)[0,1]
    ax.text(0.97,0.97,f'r = {corr:+.3f}',transform=ax.transAxes,
            ha='right',va='top',fontsize=9,color='white',fontweight='bold')

plt.suptitle('Meta-Gate Alpha vs LOCF Staleness  |  Do stale positions get more correction?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_alpha_staleness.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig5_alpha_staleness.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 6 — Time-series reconstruction (2 nodes, 12 hours)
# ════════════════════════════════════════════════════════════════════════
DEMO_SEED  = 42
DEMO_START = 4500
DEMO_LEN   = 144  # 12 hours (288 steps/day / 2)
DEMO_NODES = [10, 200]

m_demo = make_eval_mask_np(DEMO_SEED, DEMO_LEN, NUM_NODES)  # [T, N]
x_demo_np = speed_norm[DEMO_START:DEMO_START+DEMO_LEN]  # [T, N] z-scored
v_demo_np = valid_raw [DEMO_START:DEMO_START+DEMO_LEN]
gt_demo = (x_demo_np * node_stds[None,:] + node_means[None,:]).clip(0, 120)  # [T,N] km/h

# LOCF baseline
locf_demo = locf(x_demo_np.T, v_demo_np.T, m_demo.T,
                 np.arange(DEMO_START, DEMO_START+DEMO_LEN))  # [N,T] km/h

# V5 predictions
net_v5.eval()
with torch.no_grad():
    x_d  = torch.tensor(x_demo_np, device=device).T.unsqueeze(0)  # [1,N,T]
    ha_d = ha_prior[DEMO_START:DEMO_START+DEMO_LEN].T.unsqueeze(0)
    v_d  = torch.tensor(v_demo_np, device=device).T.unsqueeze(0)
    m_d  = torch.tensor(m_demo,    device=device).T.unsqueeze(0)
    m_d_eff = m_d * v_d
    ti_d = torch.arange(DEMO_START, DEMO_START+DEMO_LEN, device=device)
    ts_d = torch.sin(2*np.pi*(ti_d%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc_d = torch.cos(2*np.pi*(ti_d%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st_d = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn_d = torch.tensor(node_means, device=device).view(1,-1,1)
    preds_d = []
    for c0 in range(0, DEMO_LEN, BATCH_TIME):
        c1 = min(c0+BATCH_TIME, DEMO_LEN)
        p  = net_v5(x_d[:,:,c0:c1]*m_d_eff[:,:,c0:c1], m_d_eff[:,:,c0:c1],
                    ts_d[:,:,c0:c1], tc_d[:,:,c0:c1], ha_d[:,:,c0:c1])
        preds_d.append((p * st_d + mn_d).clamp(0, 120))
    v5_demo = torch.cat(preds_d, dim=2)[0].cpu().numpy()  # [N, T] km/h

t_ax = np.arange(DEMO_LEN) * 5 / 60  # minutes -> hours
fig, axes = plt.subplots(len(DEMO_NODES), 1, figsize=(13, 5*len(DEMO_NODES)), sharex=True)
if len(DEMO_NODES) == 1: axes = [axes]
for ax, node in zip(axes, DEMO_NODES):
    obs_mask = (m_demo[:, node] > 0) & (v_demo_np[:, node] > 0)  # [T]
    held_mask = ~obs_mask
    ax.fill_between(t_ax, 0, 120, where=held_mask, alpha=0.07,
                    color='steelblue', label='held-out (80%)')
    ax.plot(t_ax, gt_demo[:, node],   'k-',  lw=1.2, alpha=0.6, label='Ground truth')
    ax.plot(t_ax, locf_demo[node],    color='#78909C', lw=1.5, ls='--', alpha=0.85, label='LOCF')
    ax.plot(t_ax, v5_demo[node],      color='#B71C1C', lw=1.8, label='V5')
    obs_t = np.where(obs_mask)[0]
    ax.scatter(t_ax[obs_t], gt_demo[obs_t, node], s=18, c='black',
               zorder=6, label='Observed readings')
    # chunk boundary lines
    for cb in range(BATCH_TIME, DEMO_LEN, BATCH_TIME):
        ax.axvline(t_ax[cb], color='gray', lw=0.7, ls=':', alpha=0.5)
    mae_v5l  = np.abs(v5_demo[node][held_mask]   - gt_demo[:,node][held_mask]).mean()
    mae_locfl = np.abs(locf_demo[node][held_mask] - gt_demo[:,node][held_mask]).mean()
    ax.set_title(f'Sensor {node}  |  V5 MAE={mae_v5l:.2f}  LOCF MAE={mae_locfl:.2f} km/h',
                 fontweight='bold')
    ax.set_ylabel('Speed (km/h)'); ax.set_ylim(0, 120)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('Hour of demo window')
plt.suptitle(f'Time-Series Reconstruction  |  Seed {DEMO_SEED}, 12 h, dotted=chunk boundary',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_timeseries.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig6_timeseries.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 7 — MAE by hour of day (V5 vs LOCF)
# ════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 4))
H = 24
v5_hour_mae   = [tod_err_v5  [np.floor(tod_hr_v5  ).astype(int)==h].mean() for h in range(H)]
locf_hour_mae = [tod_err_locf[np.floor(tod_hr_locf).astype(int)==h].mean() for h in range(H)]
x_h = np.arange(H)
w = 0.38
ax.bar(x_h - w/2, v5_hour_mae,   width=w, color='#B71C1C', alpha=0.85, label='V5')
ax.bar(x_h + w/2, locf_hour_mae, width=w, color='#78909C', alpha=0.85, label='LOCF')
ax.set_xticks(x_h)
ax.set_xticklabels([f'{h:02d}:00' for h in range(H)], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('MAE (km/h)'); ax.set_xlabel('Hour of day')
ax.set_title('MAE by Hour of Day  |  V5 vs LOCF', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig7_mae_by_hour.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig7_mae_by_hour.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 8 — Per-node MAE scatter: V5 vs LOCF (325 sensors)
# ════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 7))
vmin = min(node_mae_v5.min(), node_mae_locf.min())
vmax = max(node_mae_v5.max(), node_mae_locf.max())
sc = ax.scatter(node_mae_locf, node_mae_v5, c=node_mae_v5,
                cmap='RdYlGn_r', vmin=vmin, vmax=vmax,
                s=30, alpha=0.75, edgecolors='none')
plt.colorbar(sc, ax=ax, label='V5 MAE (km/h)')
lim = (vmin*0.95, vmax*1.05)
ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='y = x  (no improvement)')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('LOCF MAE per sensor (km/h)')
ax.set_ylabel('V5 MAE per sensor (km/h)')
ax.set_title(f'Per-Sensor MAE: V5 vs LOCF  |  {NUM_NODES} sensors',
             fontweight='bold')
n_better = (node_mae_v5 < node_mae_locf).sum()
ax.text(0.04, 0.96,
        f'V5 better on {n_better}/{NUM_NODES} sensors ({100*n_better/NUM_NODES:.0f}%)',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('fig8_per_node_mae.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig8_per_node_mae.png')

# ════════════════════════════════════════════════════════════════════════
# Figure 9 — Error percentile bars (P50/P75/P90/P95 for top 5 models)
# ════════════════════════════════════════════════════════════════════════
# Collect per-position errors for top models
def _flat_errors(model, n_seeds=2):
    model.eval()
    out = []
    ES, EL = 4500, 450
    x_e  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_e = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_e  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti   = torch.arange(ES, ES+EL, device=device)
    ts   = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc   = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st_  = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn_  = torch.tensor(node_means, device=device).view(1,-1,1)
    with torch.no_grad():
        for seed in EVAL_SEEDS[:n_seeds]:
            m_np  = make_eval_mask_np(seed, EL, NUM_NODES)
            m_ev  = torch.tensor(m_np, device=device).T.unsqueeze(0)
            m_eff = m_ev * v_e
            for c0 in range(0, EL, BATCH_TIME):
                c1  = min(c0+BATCH_TIME, EL)
                xc  = x_e[:,:,c0:c1]; mc = m_eff[:,:,c0:c1]
                pred = model(xc*mc, mc, ts[:,:,c0:c1], tc[:,:,c0:c1], ha_e[:,:,c0:c1])
                p_km = (pred * st_ + mn_).clamp(0, 120)
                t_km = (xc   * st_ + mn_).clamp(0, 120)
                sm   = (m_ev[:,:,c0:c1]==0) & (v_e[:,:,c0:c1]>0)
                out.append(torch.abs(p_km - t_km)[sm].cpu().numpy())
    return np.concatenate(out)

print('Collecting percentile data...')
pct_data = {
    'LOCF':  locf_err_flat,
    'BRITS': _flat_errors(net if False else net),  # placeholder; reuse V3 slot
    'V3':    _flat_errors(net),
    'V4':    err_v4,
    'V5':    err_v5,
}
# Replace BRITS placeholder with actual BRITS flat errors from RESULTS mean
pct_data.pop('BRITS')
PERCS = [50, 75, 90, 95]
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(pct_data.keys())
n_m = len(model_names)
width = 0.18
colors_p = ['#42A5F5', '#66BB6A', '#FFA726', '#EF9A9A']
for pi, (pct, col) in enumerate(zip(PERCS, colors_p)):
    vals = [np.percentile(pct_data[m], pct) for m in model_names]
    x_p  = np.arange(n_m) + (pi - (len(PERCS)-1)/2) * width
    bars = ax.bar(x_p, vals, width=width, color=col, alpha=0.85,
                 label=f'P{pct}')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v:.1f}',
                ha='center', va='bottom', fontsize=7.5, rotation=90)
ax.set_xticks(np.arange(n_m))
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Absolute error (km/h)')
ax.set_title('Error Percentiles  |  P50 / P75 / P90 / P95', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig9_percentiles.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig9_percentiles.png')

print('\nAll figures saved: fig1..fig9')


# ── Fig 10: Extended Metrics Heatmap ─────────────────────────────────────────────────────────────
# Color-coded performance across all 9 metrics for each model (green = best)
fig10, ax10 = plt.subplots(figsize=(16, 4))
_mnames  = list(EXT_METRICS.keys())
_mkeys   = ["MAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE", "Hit5", "Hit10"]
_mlabels = ["MAE\n(km/h)", "RMSE\n(km/h)", "MedAE\n(km/h)", "MAPE\n(%)", "R²",
            "Pearson\nr", "MBE\n(km/h)", "Hit\n@5%", "Hit\n@10%"]
raw_mat  = np.array([[EXT_METRICS[mn][k] for k in _mkeys] for mn in _mnames])
norm_mat = np.zeros_like(raw_mat)
higher_is_better = {"R2", "Pearson", "Hit5", "Hit10"}
for j, k in enumerate(_mkeys):
    col = raw_mat[:, j]
    rng = col.max() - col.min() + 1e-9
    norm_mat[:, j] = (col - col.min()) / rng if k in higher_is_better else 1 - (col - col.min()) / rng
im10 = ax10.imshow(norm_mat, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax10.set_xticks(range(len(_mkeys)));  ax10.set_xticklabels(_mlabels, fontsize=10)
ax10.set_yticks(range(len(_mnames))); ax10.set_yticklabels([n.split(". ", 1)[-1] for n in _mnames], fontsize=11)
for i in range(len(_mnames)):
    for j, k in enumerate(_mkeys):
        v = raw_mat[i, j]
        fmt = f"{v:.3f}" if k in ("R2", "Pearson") else (f"{v:.2f}" if k in ("MAPE", "Hit5", "Hit10") else f"{v:.3f}")
        ax10.text(j, i, fmt, ha="center", va="center", fontsize=8.5,
                  color="black" if 0.25 < norm_mat[i, j] < 0.78 else "white")
plt.colorbar(im10, ax=ax10, label="Normalized Score (1 = best)", fraction=0.02, pad=0.01)
ax10.set_title("Extended Metrics Heatmap — PEMS-BAY  (green = best per column)", fontsize=13, pad=10)
plt.tight_layout()
fig10.savefig("fig10_metrics_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 11: MAE by Speed Regime ──────────────────────────────────────────────────────────────────
# Compare models in slow / medium / fast traffic conditions
fig11, ax11 = plt.subplots(figsize=(11, 5))
regimes   = [("Low  (<30 km/h)", 0, 30), ("Medium  (30-60)", 30, 60), ("High  (>60 km/h)", 60, 130)]
rlabels   = [r[0] for r in regimes]
m_reg     = {"LOCF": (p_lc, t_lc), "V3": (p_v3, t_v3), "V4": (p_v4, t_v4), "V5 (ours)": (p_v5, t_v5)}
c_reg     = ["#2196F3", "#FF9800", "#9C27B0", "#4CAF50"]
xpos      = np.arange(len(regimes));  bw = 0.19
for ki, (name, (pp, tt)) in enumerate(m_reg.items()):
    maes = []
    for (_, lo, hi) in regimes:
        mk = (tt >= lo) & (tt < hi)
        maes.append(float(np.abs(pp[mk] - tt[mk]).mean()) if mk.sum() > 0 else 0.0)
    bars = ax11.bar(xpos + ki * bw, maes, bw, label=name, color=c_reg[ki], alpha=0.88, edgecolor="white")
    for bar, v in zip(bars, maes):
        ax11.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, f"{v:.2f}",
                  ha="center", va="bottom", fontsize=8.5, color=c_reg[ki])
ax11.set_xticks(xpos + bw * 1.5); ax11.set_xticklabels(rlabels, fontsize=11)
ax11.set_ylabel("MAE (km/h)", fontsize=12); ax11.grid(axis="y", alpha=0.3)
ax11.set_title("MAE by Speed Regime — Low / Medium / High Traffic", fontsize=13)
ax11.legend(fontsize=11); plt.tight_layout()
fig11.savefig("fig11_speed_regime_mae.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 12: Prediction Density Scatter (V5 vs LOCF) ─────────────────────────────────────────────
fig12, axes12 = plt.subplots(1, 2, figsize=(14, 6))
for ax12, (name, pp, tt, r2key) in zip(axes12, [
        ("V5 (ours)", p_v5, t_v5, "5. V5 (ours)"),
        ("LOCF",      p_lc, t_lc, "2. LOCF")]):
    hb = ax12.hexbin(tt, pp, gridsize=65, cmap="viridis", mincnt=1, bins="log")
    ax12.plot([0, 120], [0, 120], "r--", lw=1.5, label="Perfect (y=x)")
    ax12.set_xlabel("Ground Truth (km/h)", fontsize=11)
    ax12.set_ylabel("Prediction (km/h)", fontsize=11)
    ax12.set_title(f"{name} — Prediction Density", fontsize=12)
    ax12.set_xlim(0, 120); ax12.set_ylim(0, 120)
    ax12.legend(fontsize=10)
    plt.colorbar(hb, ax=ax12, label="log(count)")
    rv = EXT_METRICS[r2key]
    ax12.text(4, 112, f"R²={rv['R2']:.4f}   Pearson={rv['Pearson']:.4f}",
              fontsize=10, color="white", bbox=dict(facecolor="#222", alpha=0.75, pad=3))
plt.suptitle("Prediction Density: V5 vs LOCF", fontsize=14)
plt.tight_layout()
fig12.savefig("fig12_pred_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 13: Hit-Rate Curves ───────────────────────────────────────────────────────────────────────
# % of predictions within tolerance eps for eps in [0, 20] km/h
fig13, ax13 = plt.subplots(figsize=(10, 6))
thresholds = np.linspace(0, 20, 250)
hr_models  = {"HA": (p_ha, t_ha, "#9E9E9E", "--", 1.5),
              "LOCF": (p_lc, t_lc, "#2196F3", "-",  2.0),
              "V3":   (p_v3, t_v3, "#FF9800", "--", 1.5),
              "V4":   (p_v4, t_v4, "#9C27B0", "-",  1.8),
              "V5 (ours)": (p_v5, t_v5, "#4CAF50", "-", 2.5)}
for name, (pp, tt, col, ls, lw) in hr_models.items():
    errs = np.abs(pp - tt)
    ax13.plot(thresholds, [(errs < thr).mean()*100 for thr in thresholds],
              label=name, color=col, ls=ls, lw=lw)
for vx, lbl in [(5, "5 km/h"), (10, "10 km/h")]:
    ax13.axvline(vx, color="gray", ls=":", lw=1, alpha=0.6)
    ax13.text(vx + 0.2, 4, lbl, fontsize=9, color="gray")
ax13.set_xlabel("Error Tolerance ε (km/h)", fontsize=12)
ax13.set_ylabel("Hit Rate — % predictions within ε", fontsize=12)
ax13.set_title("Hit-Rate Curves: Cumulative Accuracy at Every Tolerance", fontsize=13)
ax13.legend(fontsize=11); ax13.grid(alpha=0.3)
ax13.set_xlim(0, 20); ax13.set_ylim(0, 100)
plt.tight_layout()
fig13.savefig("fig13_hit_rate_curves.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 14: RMSE vs MAE — Outlier Sensitivity ────────────────────────────────────────────────────
fig14, ax14 = plt.subplots(figsize=(8, 6))
to_models = {"HA": m_ha, "LOCF": m_lc, "V3": m_v3, "V4": m_v4, "V5 (ours)": m_v5}
c_to      = {"HA": "#9E9E9E", "LOCF": "#2196F3", "V3": "#FF9800", "V4": "#9C27B0", "V5 (ours)": "#4CAF50"}
mk_to     = {"HA": "s",       "LOCF": "D",        "V3": "o",       "V4": "^",       "V5 (ours)": "*"}
for name, m in to_models.items():
    ax14.scatter(m["MAE"], m["RMSE"], s=220 if "V5" in name else 130,
                 color=c_to[name], marker=mk_to[name], zorder=5,
                 edgecolors="white", linewidths=1.2, label=name)
    ax14.annotate(f"  {name}\n  ratio={m['RMSE']/m['MAE']:.3f}",
                  (m["MAE"], m["RMSE"]), fontsize=8.5)
mae_lo = min(m["MAE"] for m in to_models.values()) - 0.05
mae_hi = max(m["MAE"] for m in to_models.values()) + 0.05
ax14.plot([mae_lo, mae_hi], [mae_lo, mae_hi], "k--", lw=1, alpha=0.4, label="RMSE = MAE")
ax14.set_xlabel("MAE (km/h)", fontsize=12); ax14.set_ylabel("RMSE (km/h)", fontsize=12)
ax14.set_title("RMSE vs MAE — Outlier Sensitivity  (RMSE/MAE ratio annotated)", fontsize=13)
ax14.legend(fontsize=10); ax14.grid(alpha=0.3)
plt.tight_layout()
fig14.savefig("fig14_rmse_vs_mae.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 15: Error Bias by Speed Decile ───────────────────────────────────────────────────────────
# Mean bias (pred - true) and MAE across 10 ground-truth speed deciles
fig15, axes15 = plt.subplots(1, 2, figsize=(15, 5))
deciles   = np.percentile(t_v5, np.linspace(0, 100, 11))
centers   = 0.5 * (deciles[:-1] + deciles[1:])
bd_models = {"LOCF": (p_lc, t_lc), "V3": (p_v3, t_v3),
             "V4": (p_v4, t_v4), "V5 (ours)": (p_v5, t_v5)}
c_bd      = {"LOCF": "#2196F3", "V3": "#FF9800", "V4": "#9C27B0", "V5 (ours)": "#4CAF50"}
for ax_b, (fn, ylabel, title_suf) in zip(axes15, [
        (lambda e: e.mean(),      "Mean Bias (km/h)",  "Prediction Bias"),
        (lambda e: np.abs(e).mean(), "MAE (km/h)",     "MAE")]):
    for name, (pp, tt) in bd_models.items():
        vals = []
        for lo, hi in zip(deciles[:-1], deciles[1:]):
            mk = (tt >= lo) & (tt <= hi)
            vals.append(fn(pp[mk] - tt[mk]) if mk.sum() > 0 else np.nan)
        ax_b.plot(centers, vals, "o-", color=c_bd[name], label=name, lw=2, ms=5)
    if ylabel.startswith("Mean"):
        ax_b.axhline(0, color="k", ls="--", lw=1, alpha=0.5)
    ax_b.set_xlabel("Ground Truth Speed (km/h)", fontsize=11)
    ax_b.set_ylabel(ylabel, fontsize=11)
    ax_b.set_title(f"{title_suf} by Speed Decile", fontsize=12)
    ax_b.legend(fontsize=10); ax_b.grid(alpha=0.3)
plt.suptitle("Error Profile Across Speed Deciles", fontsize=14, y=1.02)
plt.tight_layout()
fig15.savefig("fig15_bias_by_speed_decile.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 16: Improvement Radar over Baselines ─────────────────────────────────────────────────────
# Radar showing V3/V4/V5 fractional improvement over HA and LOCF across 5 error dimensions
fig16, axes16 = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(polar=True))
rad_keys   = ["MAE", "RMSE", "MedAE", "MAPE", "Hit10"]
rad_labels = ["MAE", "RMSE", "MedAE", "MAPE%", "Hit@10%"]
N_r = len(rad_keys)
angles = np.linspace(0, 2*np.pi, N_r, endpoint=False).tolist() + [0]
def _rv(m): return [m[k] for k in rad_keys]
for ax_r, (bname, bm) in zip(axes16, [("HA", m_ha), ("LOCF", m_lc)]):
    bvals = _rv(bm)
    for (vname, vm), col in zip([("V3", m_v3), ("V4", m_v4), ("V5 (ours)", m_v5)],
                                 ["#FF9800", "#9C27B0", "#4CAF50"]):
        vvals = _rv(vm)
        # Improvement = (baseline - model) / baseline; for Hit10 it's (model - baseline) / baseline
        scores = [(b - v) / (abs(b) + 1e-9) if k != "Hit10" else (v - b) / (abs(b) + 1e-9)
                  for k, v, b in zip(rad_keys, vvals, bvals)]
        s_plot = scores + scores[:1]
        ax_r.plot(angles, s_plot, "o-", lw=2, color=col, label=vname)
        ax_r.fill(angles, s_plot, alpha=0.12, color=col)
    ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(rad_labels, fontsize=11)
    ax_r.axhline(0, color="gray", lw=0.8, ls="--")
    ax_r.set_title(f"Improvement over {bname}", size=13, pad=16)
    ax_r.legend(loc="upper right", bbox_to_anchor=(1.4, 1.12), fontsize=10)
    ax_r.set_ylim(-0.25, 0.45)
plt.suptitle("V-Series Improvement Radar (positive = better than baseline)", fontsize=13, y=1.03)
plt.tight_layout()
fig16.savefig("fig16_radar_improvement.png", dpi=150, bbox_inches="tight")
plt.show()


# ── Fig 17: Cumulative Per-Node Improvement (V5 vs LOCF) ─────────────────────────────────────────
# Sorted sensor-level MAE improvement — shows breadth and magnitude of gains
fig17, ax17 = plt.subplots(figsize=(12, 5))
def _per_node_mae_ext(pp_flat, tt_flat, n_nodes=NUM_NODES):
    chunk = len(pp_flat) // n_nodes
    return np.array([np.abs(pp_flat[i*chunk:(i+1)*chunk] - tt_flat[i*chunk:(i+1)*chunk]).mean()
                     for i in range(n_nodes)])
# Per-node split only works cleanly when samples divide evenly; use sorted global approach instead
node_mae_lc = np.zeros(NUM_NODES);  node_mae_v5 = np.zeros(NUM_NODES)
node_mae_v4 = np.zeros(NUM_NODES);  node_mae_v3 = np.zeros(NUM_NODES)
# Recompute per-node by running eval on a single seed with node-indexed output
with torch.no_grad():
    _x  = speed_gpu[4500:4950].T.unsqueeze(0)
    _ha = ha_prior [4500:4950].T.unsqueeze(0)
    _v  = valid_gpu[4500:4950].T.unsqueeze(0)
    _t  = torch.arange(4500, 4950, device=device)
    _ts = torch.sin(2*np.pi*(_t%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    _tc = torch.cos(2*np.pi*(_t%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    _st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    _mn = torch.tensor(node_means, device=device).view(1,-1,1)
    _mnp   = make_eval_mask_np(42, 450, NUM_NODES)
    _meval = torch.tensor(_mnp, device=device).T.unsqueeze(0)
    _meff  = _meval * _v
    # LOCF
    _prev = _ha[:,:,0:1].clone(); _lout = []
    for _ti in range(450):
        _obs = _x[:,:,_ti:_ti+1] * _meff[:,:,_ti:_ti+1]
        _prev = torch.where(_meff[:,:,_ti:_ti+1]>0, _obs, _prev)
        _lout.append(_prev.clone())
    _plc = torch.cat(_lout, dim=2)
    # Models (chunked)
    def _fwd(mdl):
        _ps = []
        for _c0 in range(0, 450, BATCH_TIME):
            _c1 = min(_c0+BATCH_TIME, 450)
            _ps.append(mdl(_x[:,:,_c0:_c1]*_meff[:,:,_c0:_c1], _meff[:,:,_c0:_c1],
                           _ts[:,:,_c0:_c1], _tc[:,:,_c0:_c1], _ha[:,:,_c0:_c1]))
        return torch.cat(_ps, dim=2)
    _pv3 = _fwd(net);  _pv4 = _fwd(net_v4);  _pv5 = _fwd(net_v5)
    _tk  = (_x  * _st + _mn).clamp(0, 120)
    for _mi, pp in zip([node_mae_lc, node_mae_v3, node_mae_v4, node_mae_v5],
                        [_plc, _pv3, _pv4, _pv5]):
        _pk = (pp * _st + _mn).clamp(0, 120)
        _sm_n = (_meval==0) & (_v>0)
        for ni in range(NUM_NODES):
            _s  = _sm_n[0, ni]
            if _s.sum() > 0:
                _mi[ni] = torch.abs(_pk[0, ni][_s] - _tk[0, ni][_s]).mean().item()

_imp_v5   = node_mae_lc - node_mae_v5
_imp_v4   = node_mae_lc - node_mae_v4
_sort_idx = np.argsort(_imp_v5)[::-1]
ax17.bar(range(NUM_NODES), _imp_v5[_sort_idx], color="#4CAF50", alpha=0.7, label="V5 vs LOCF")
ax17.bar(range(NUM_NODES), _imp_v4[_sort_idx], color="#9C27B0", alpha=0.5, label="V4 vs LOCF", width=0.6)
ax17.axhline(0, color="black", lw=0.8)
ax17.set_xlabel("Sensor Rank (sorted by V5 improvement)", fontsize=11)
ax17.set_ylabel("MAE improvement over LOCF (km/h)", fontsize=11)
ax17.set_title("Per-Sensor Imputation Improvement: V4 and V5 vs LOCF  (325 sensors)", fontsize=13)
ax17.legend(fontsize=11); ax17.grid(axis="y", alpha=0.3)
frac_pos = (_imp_v5 > 0).mean() * 100
ax17.text(0.02, 0.96, f"V5 improves {frac_pos:.0f}% of sensors over LOCF",
          transform=ax17.transAxes, fontsize=11, va="top",
          bbox=dict(facecolor="white", alpha=0.8, edgecolor="#4CAF50"))
plt.tight_layout()
fig17.savefig("fig17_per_sensor_improvement.png", dpi=150, bbox_inches="tight")
plt.show()
